# Energy-classified building dataset (version 5)

**Goal of the challenge (Building Footprints to Energy Demand):** for every building in the area, estimate
its building type, yearly electricity use (kWh), peak demand (kW), daily load pattern, cooling demand
category and productive-use potential, and save it as one map file.

This notebook starts from the roof map Eliane's model already made (`full_satellite_prediction.tif`).

**New in version 5 (after the 12 October call with Eliane):**
- Section 9 now gives the **hourly load profile (hour 0-23)** and a **monthly load profile** linked to the
  NASA temperatures (hotter months use more).
- New section 12, **Area explorer**: a map cut into squares; click a square to see its buildings, types,
  yearly use, average day and month-by-month use.
- Section 13 has a **placeholder register**: every guessed number, and the source we still need to cite for it.
- Productive use (section 8) stays, but light: there is a separate hackathon challenge on productive use.

| Step | What it adds to the table |
|---|---|
| 1. Setup | installs the tools, connects Google Drive, fixes RAMP |
| 2. Settings | every number you may want to change, in one place |
| 3. Roofs to buildings | one outline per building, with its size |
| 3b. Check | how well the model agrees with Overture's building map |
| 3c. Split | fixes buildings the model stuck together |
| 4. Building type | school, clinic, shop, church, workshop or home |
| 4b. Roads | distance from each building to the nearest (main) road |
| 4c. Outbuildings | small kitchens, toilets and stores next to a home |
| 5. Homes' electricity | households, yearly kWh, peak, daily pattern (from survey-based household profiles (Nigeria, RAMP), adjusted for roads) |
| 6. Other buildings' electricity (RAMP) | yearly kWh, peak and pattern for schools, clinics, shops... from appliance lists |
| 7. Weather | cooling demand category (Low / Medium / High) |
| 8. Productive use | could this building host a business (High / Medium / Low) and why |
| 9. Load profiles | how steady each building's use is; the whole area hour by hour and month by month |
| 10. Summary | totals by type; the answer to Eliane's weather-and-roads question |
| 11. Save and map | the deliverable file and an interactive map |
| 12. Area explorer | clickable map: numbers, daily and monthly profile for any part of the area |
| 13. Next steps | decisions from the call, placeholder register, open questions, references |

**To run:** Colab menu → **Runtime** → **Restart session and run all**. Approve the Google Drive pop-up.
It takes about 10 minutes. No graphics card needed.

**To change something:** edit the number in **Settings** (or the appliance lists in **6a**), then
Runtime → Restart session and run all again.

Words used below:
- **kWh** (kilowatt-hour): an amount of electricity. A 100-watt bulb on for 10 hours uses 1 kWh.
- **kW** (kilowatt): how much is being used at one moment. **Peak** = the highest it gets.
- **Load profile / load pattern**: how use goes up and down over the day.
- **Placeholder**: a reasonable first guess that still needs to be confirmed.

## Data sources

| Dataset | What it gives us | Why we chose it | Licence / citation |
|---|---|---|---|
| 30 cm satellite image (kenya_preflood.tif) | The photo the roof-detection model reads | Sharp enough to see single houses | Maxar GeoEye-1, 11 May 2024, 0.31 m, via OpenAerialMap / HOT (image 6642a8a66049ef00013b85d7), licence CC BY-NC 4.0. Cropped for pipeline development. |
| Overture Maps – Buildings (release 2026-09-23.1) | Building outlines (no building types) | Free and global; merges Google Open Buildings, Microsoft ML Building Footprints and OpenStreetMap | ODbL. Overture Maps Foundation; Sirko et al. 2021 (Google Open Buildings); Microsoft Global ML Building Footprints |
| OpenStreetMap (Overpass API) | Building-type labels (school, clinic, shop...) | Only open source of building-use labels | ODbL. © OpenStreetMap contributors |
| Overture Maps – Places (release 2026-09-23.1) | More labels: businesses and services | Adds shops and services OpenStreetMap lacks | CDLA Permissive 2.0. Overture Maps Foundation |
| Overture Maps – Transportation (release 2026-09-23.1) | Road network and road class | Distance to main roads, our infrastructure signal; more reliable than the Overpass servers | ODbL. Overture Maps Foundation (based on OpenStreetMap) |
| Household load profiles | 24-hour use patterns for homes | Best available stand-in; real metered Kenyan homes not available | RAMP-simulated profiles from a household survey in Nigeria (project dataset); not measured in Kenya. |
| Training labels | Building outlines drawn by hand in QGIS (by Eliane) | Ground truth to train the roof-detection model | Project data |
| NASA POWER (daily temperature, 2023) | Temperature at the site | Free, global, daily; named in the challenge README | NASA Langley Research Center POWER Project |
| RAMP (tool, not data) | Load profiles for schools, clinics, shops | Open-source tool the challenge asks for | Lombardi et al. 2019, Energy 177:433-444 |
| Land use / zoning | Not used yet | Suggested by reviewers; see Next steps | - |

Note: building_ke2.json (OpenStreetMap) was not used in training.

## 1. Setup

Installs the libraries (including **RAMP**, the tool that builds electricity profiles from appliance lists),
connects Google Drive, and then applies a small fix so RAMP works with the newer Python that Colab uses.

In [ ]:
# Installs the libraries (including RAMP, the tool that builds electricity profiles from appliance lists).
%pip install rasterio geopandas shapely pyproj mapclassify folium matplotlib scikit-learn opencv-python-headless requests pyarrow rampdemand overturemaps

In [ ]:
# Makes the RAMP package work with the newer numpy and pandas that Colab uses.
# It edits two small things inside RAMP's own code. Safe to run more than once.
import importlib.util, os, re

spec = importlib.util.find_spec("ramp")                     # finds RAMP without loading it
assert spec is not None, "RAMP is not installed: run the Setup cell first."
core_path = os.path.join(os.path.dirname(spec.origin), "core", "core.py")
src = open(core_path).read()
fixed = re.sub(r"np\.diff\(self\.window_(\d)\)(?!\[0\])", r"np.diff(self.window_\1)[0]", src)   # numpy 2 fix
fixed = (fixed.replace('pd.Timedelta(1, "T")', 'pd.Timedelta(1, "min")')                     # pandas fix
              .replace('freq="T"', 'freq="min"'))
if fixed != src:
    open(core_path, "w").write(fixed)
    print("RAMP fixed for this Colab.")
else:
    print("RAMP already fixed.")

## 2. Settings

This cell finds Eliane's folder by itself and prints **FOUND** or **NOT FOUND**.
If NOT FOUND: in Google Drive open "Shared with me", right-click her folder → Organize → Add shortcut →
My Drive. Then Restart session and run all.

Three groups of numbers:
1. **From Eliane's notebook**: do not change without her.
2. **Placeholders from earlier versions**.
3. **New placeholders in this version** (outbuildings, roads, RAMP, weather, productive use).

Every placeholder is a question for Eliane or the experts (see section 12).

In [ ]:
# ---- Where the files are -------------------------------------------------
import glob, os
MASK_NAME = "full_satellite_prediction.tif"              # the model's saved roof map
BASE = r"/Users/iiyam112156/Desktop/Temp_pipeline"
MASK_PATH = os.path.join(BASE, MASK_NAME)
if os.path.exists(MASK_PATH):
    print("FOUND:", MASK_PATH)
else:
    print("NOT FOUND (information only, Overture outlines will be used instead):", MASK_PATH)

# Where building outlines come from: the model's roof map if it exists, otherwise Overture's outlines.
FOOTPRINT_SOURCE = "model" if (MASK_PATH and os.path.exists(MASK_PATH)) else "overture"
OVERTURE_BUILDINGS = os.path.join(BASE, "overture_download", "buildings.parquet")
OVERTURE_PLACES = os.path.join(BASE, "overture_download", "places.parquet")
print("Building outlines from:", FOOTPRINT_SOURCE)
PROFILE_CSV = ("https://docs.google.com/spreadsheets/d/"
               "1XfBXKTcOGwi4xoOQcVIzEv92QWdRqdp2pREuF-LgYp4/export?format=csv")  # survey-based household load profiles (Nigeria, RAMP)
WORK_DIR = os.path.abspath("building_table_out")         # results are written here first
DRIVE_OUT_DIR = None

# ---- 1) Copied from Eliane's notebook (do not change without her) ----------
MIN_AREA_PX = 80             # ignore blobs smaller than this many pixels
MAX_ASPECT_RATIO = 4.0       # ignore very long thin blobs (usually roads or walls)
COMPOUND_DISTANCE_PX = 30    # roofs closer than this belong to the same household
N_ARCHETYPES = 4             # number of typical daily patterns
MONTHLY_KWH_MEAN = 78.0 * 0.8   # average household use per month (62.4 kWh; Kenya survey average is 48.6)
ALPHA = 0.8                  # how strongly use grows with roof size
BUILDING_WEIGHT = 0.15       # extra use for each additional building in a compound
ROOF_CLIP_QUANTILE = 0.98    # stop a few huge roofs from dominating
SIGMA_HOUSEHOLD = 0.35       # random household-to-household variation
SEED = 42                    # makes the random parts give the same result every run

# ---- 2) Placeholders from earlier versions ---------------------------------
LARGE_BUILDING_M2 = 400      # unlabelled buildings bigger than this are NOT assumed to be homes
POI_MAX_DISTANCE_M = 15      # how far a map label may sit from the building it describes
DAYS_SAMPLED = 7             # days of readings sampled per household to find its peak
MIN_OVERLAP = 0.30           # an Overture building is 'inside' a model shape if 30%+ of it overlaps
INCLUDE_MISSED = True        # add the Overture buildings the model missed (mostly small)
PLACES_MIN_CONFIDENCE = 0.5  # Overture Places below this confidence are ignored

# ---- 3) New placeholders in this version -----------------------------------
OUTBUILDING_MAX_M2 = 50      # Eliane: homes are above 50-80 m2. Smaller buildings next to a home = outbuilding
ATTACH_DISTANCE_M = 15       # ...if a bigger home is within this many metres
ROAD_EFFECT = 0.20           # households closest to a main road use +20%, farthest -20% (area total unchanged)
RAMP_DAYS = 28               # days RAMP simulates per building type
RAMP_START = "2023-01-02"    # first simulated day (a Monday)
SIZE_EXPONENT = 0.8          # non-home buildings: double the size -> about 1.7x the electricity
SCALE_LIMITS = (0.3, 5.0)    # ...but never below 0.3x or above 5x the typical building in 6a
WEATHER_YEAR = 2023          # year of temperature data from NASA POWER
COOLING_MEDIUM_FROM = 500    # cooling degree days per year at which cooling need becomes Medium
COOLING_HIGH_FROM = 1500     # ...and High
PUE_HOME_MIN_M2 = 100        # productive use: a home this big near a main road could host a business
PUE_ROAD_M = 250             # ...'near' means within this many metres of a main road
TEMP_SENSITIVITY = 0.03      # monthly profile: +3% use for each degree C a month is warmer than the yearly average
CELL_M = 250                 # area explorer: size of each square, in metres

## 3. Roofs to buildings

**Adds:** one outline per building, its size (`area_m2`), an ID and its location (`lon`, `lat`).

The roof map is a grid of pixels: 1 = roof, 0 = not roof. This traces around each group of touching
roof pixels. Two buildings that touch come out as one shape; step 3c fixes that.

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import cv2
from rasterio.features import shapes
from shapely.geometry import shape

# Skipped when there is no roof map; section 3d builds the outlines from Overture instead.
if FOOTPRINT_SOURCE == "model":
    with rasterio.open(MASK_PATH) as src:
        mask = (src.read(1) > 0).astype(np.uint8)
        transform, crs = src.transform, src.crs

    assert crs is not None and crs.is_projected, "The prediction file must use a metre-based map projection."
    pixel_m = abs(transform.a)           # width of one pixel in metres
    pixel_area_m2 = abs(transform.a * transform.e)

    # Same clean-up Eliane uses: remove specks, then close small gaps.
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # Trace each group of touching roof pixels into an outline with real map coordinates.
    outlines = [shape(geom) for geom, value in
                shapes(mask, mask=mask.astype(bool), transform=transform, connectivity=8) if value == 1]
    buildings = gpd.GeoDataFrame(geometry=outlines, crs=crs)
    n_traced = len(buildings)

    # Apply Eliane's two filters: too small, or too long and thin.
    buildings["area_m2"] = buildings.geometry.area
    b = buildings.geometry.bounds
    width, height = b["maxx"] - b["minx"], b["maxy"] - b["miny"]
    aspect = np.maximum(width, height) / (np.minimum(width, height) + 1e-6)
    keep = (buildings["area_m2"] >= MIN_AREA_PX * pixel_area_m2) & (aspect <= MAX_ASPECT_RATIO)
    buildings = buildings[keep].reset_index(drop=True)

    # Smooth the stair-step pixel edges a little so the file is smaller and looks cleaner.
    buildings["geometry"] = buildings.geometry.simplify(pixel_m, preserve_topology=True)

    buildings["building_id"] = [f"B{i:05d}" for i in range(1, len(buildings) + 1)]
    centres = buildings.geometry.centroid.to_crs(4326)
    buildings["lon"], buildings["lat"] = centres.x, centres.y

    print(f"Pixel size: {pixel_m:.3f} m")
    print(f"Shapes traced: {n_traced}  ->  buildings kept: {len(buildings)}")
    print(buildings["area_m2"].describe().round(1))

## 3b. Check the model against Overture

**Adds:** a quality check (no new columns). Overture is a free, existing map of building outlines.
This counts how many buildings the model found, missed, merged, or found that Overture does not have.
The map shows green = matches, purple = merged, orange = model only, red outline = missed.

In [ ]:
# 3b. Check the model against Overture's building outlines.
# Skipped when there is no roof map (nothing to check against).
if FOOTPRINT_SOURCE == "model":
    from shapely.geometry import box
    from rasterio.transform import array_bounds

    # Overture outlines for the same area (the two files Eliane downloaded)
    ovt_files = [os.path.join(BASE, "OvertureMaps", "building1.shp"), os.path.join(BASE, "OvertureMaps", "building2.shp")]
    overture = pd.concat([gpd.read_file(p).to_crs(crs) for p in ovt_files], ignore_index=True)
    west_m, south_m, east_m, north_m = array_bounds(mask.shape[0], mask.shape[1], transform)
    image_box = box(west_m, south_m, east_m, north_m)
    overture = overture[overture.intersects(image_box)].copy()
    overture = overture.drop_duplicates(subset="id") if "id" in overture else overture
    overture["ovt_area_m2"] = overture.geometry.area
    overture = overture.reset_index(drop=True)
    overture["ovt_index"] = overture.index

    # Which model buildings touch which Overture buildings
    pairs = gpd.sjoin(buildings[["building_id", "geometry"]], overture[["ovt_index", "geometry"]],
                      how="inner", predicate="intersects")
    ovt_per_building = pairs.groupby("building_id")["ovt_index"].nunique()
    buildings["overture_matches"] = buildings["building_id"].map(ovt_per_building).fillna(0).astype(int)
    found_ovt = set(pairs["ovt_index"])
    overture["found_by_model"] = overture["ovt_index"].isin(found_ovt)

    min_area_m2 = MIN_AREA_PX * pixel_area_m2
    big_enough = overture["ovt_area_m2"] >= min_area_m2

    n_model, n_ovt = len(buildings), len(overture)
    n_new = int((buildings["overture_matches"] == 0).sum())
    n_merged = int((buildings["overture_matches"] >= 2).sum())
    n_missed = int((~overture["found_by_model"]).sum())
    n_missed_big = int((~overture["found_by_model"] & big_enough).sum())

    # Roof area agreement: how much of the roof area the two maps share
    model_roofs = buildings.geometry.union_all() if hasattr(buildings.geometry, "union_all") else buildings.geometry.unary_union
    ovt_roofs = overture.geometry.union_all() if hasattr(overture.geometry, "union_all") else overture.geometry.unary_union
    shared = model_roofs.intersection(ovt_roofs).area
    overlap_iou = shared / model_roofs.union(ovt_roofs).area

    print(f"Model buildings: {n_model}   |   Overture buildings in the image: {n_ovt}")
    print(f"Overture buildings the model found:   {n_ovt - n_missed} ({(n_ovt - n_missed) / n_ovt:.0%})")
    print(f"Overture buildings the model missed:  {n_missed}  (of which {n_missed_big} are at least {min_area_m2:.1f} m2)")
    print(f"Model buildings not in Overture:      {n_new}  <- possibly new buildings, or false alarms")
    print(f"Model buildings covering 2+ Overture buildings: {n_merged}  <- touching buildings merged into one")
    print(f"Roof area agreement (overlap / combined area): {overlap_iou:.0%}")
    print()
    print("Size of the missed Overture buildings (m2):")
    print(overture.loc[~overture["found_by_model"], "ovt_area_m2"].describe().round(1).to_string())

    # Map: green = matches Overture, purple = merged, orange = model only, red outline = missed by the model
    view = buildings.assign(check=np.where(buildings["overture_matches"] == 0, "model only (new?)",
                                           np.where(buildings["overture_matches"] >= 2, "merged", "matches Overture")))
    check_map = view[["building_id", "area_m2", "check", "geometry"]].to_crs(4326).explore(
        column="check", categorical=True, cmap=["#2a9d8f", "#7b2cbf", "#f4a261"],
        tiles="Esri.WorldImagery", style_kwds={"weight": 1, "fillOpacity": 0.5}, name="model buildings")
    overture.loc[~overture["found_by_model"], ["ovt_area_m2", "geometry"]].to_crs(4326).explore(
        m=check_map, color="red", style_kwds={"fill": False, "weight": 2}, name="missed by model")
    os.makedirs(WORK_DIR, exist_ok=True)
    check_map.save(os.path.join(WORK_DIR, "model_vs_overture.html"))
    check_map

## 3c. Split merged buildings

**Adds:** `outline_source` and `new_candidate`. Where one model shape covers two or more Overture buildings
(each at least 30% inside it), the model shape is replaced by Overture's separate outlines.
Buildings only the model found are kept and marked `new_candidate = True` (new building, or a mistake).

In [ ]:
# 3c. Split merged buildings using Overture's outlines (no retraining needed).
# Skipped when there is no roof map; section 3d produces the outlines instead.
if FOOTPRINT_SOURCE == "model":
    import shapely

    # Keep the model's original shapes the first time this cell runs, so it can be re-run safely.
    if "buildings_model" not in globals():
        buildings_model = buildings[["building_id", "area_m2", "geometry"]].copy()

    model = buildings_model.copy()
    ovt = overture[["ovt_index", "ovt_area_m2", "geometry"]].copy()
    ovt["geometry"] = shapely.force_2d(ovt.geometry.values)

    # How much of each Overture building lies inside each model shape it touches
    pairs = gpd.sjoin(ovt, model[["building_id", "geometry"]], how="inner", predicate="intersects")
    pairs["overlap_m2"] = shapely.area(shapely.intersection(
        ovt.geometry.loc[pairs.index].values,
        model.set_index("building_id").loc[pairs["building_id"], "geometry"].values))
    pairs["overlap_frac"] = pairs["overlap_m2"] / pairs["ovt_area_m2"]

    # Each Overture building belongs to the model shape that covers most of it (if 30%+)
    best = (pairs[pairs["overlap_frac"] >= MIN_OVERLAP]
            .sort_values("overlap_frac", ascending=False)
            .drop_duplicates("ovt_index"))
    inside = best.groupby("building_id")["ovt_index"].apply(list)
    n_inside = model["building_id"].map(inside.apply(len)).fillna(0).astype(int)

    keep_single = model[n_inside == 1].assign(outline_source="model", new_candidate=False)
    new_only = model[n_inside == 0].assign(outline_source="model_new", new_candidate=True)
    split_ids = [i for ids in inside[inside.apply(len) >= 2] for i in ids]
    split = ovt[ovt["ovt_index"].isin(split_ids)].assign(outline_source="overture_split", new_candidate=False)
    missed = ovt[~ovt["ovt_index"].isin(best["ovt_index"])].assign(outline_source="overture_missed", new_candidate=False)

    parts = [keep_single, new_only, split] + ([missed] if INCLUDE_MISSED else [])
    cols = ["outline_source", "new_candidate", "geometry"]
    buildings = gpd.GeoDataFrame(pd.concat([p[cols] for p in parts], ignore_index=True), crs=crs)
    buildings["area_m2"] = buildings.geometry.area
    buildings["building_id"] = [f"B{i:05d}" for i in range(1, len(buildings) + 1)]
    centres = buildings.geometry.centroid.to_crs(4326)
    buildings["lon"], buildings["lat"] = centres.x, centres.y

    print(f"Model shapes before: {len(model)}")
    print(f"  kept as one building:            {len(keep_single)}")
    print(f"  merged shapes split up:          {int((n_inside >= 2).sum())}  -> replaced by {len(split)} Overture outlines")
    print(f"  model-only (possible new):       {len(new_only)}")
    print(f"Overture buildings the model missed, added: {len(missed) if INCLUDE_MISSED else 0}")
    print(f"Buildings now: {len(buildings)}")
    print()
    print(buildings.groupby("outline_source")["area_m2"].describe()[["count", "50%", "max"]].round(1))

## 3d. Building outlines from Overture (used when the roof map is not available)

When the model's roof map (`full_satellite_prediction.tif`) is missing, this cell builds the `buildings`
table from Overture's building outlines downloaded for the study area, which combine Google Open Buildings,
Microsoft ML Buildings and OpenStreetMap. It stands in for sections 3, 3b and 3c, producing the same
`buildings` columns so the rest of the notebook runs unchanged.

In [ ]:
# 3d. Building outlines from Overture (used when the roof map is not available).
if FOOTPRINT_SOURCE == "overture":
    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import shapely
    from pyproj import CRS

    crs = CRS.from_epsg(32737)          # UTM zone 37S, metres
    pixel_m = 0.3061
    pixel_area_m2 = pixel_m ** 2

    raw = gpd.read_parquet(OVERTURE_BUILDINGS)
    raw = raw[raw.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    if raw.crs is None:
        raw = raw.set_crs(4326)
    raw["geometry"] = shapely.force_2d(raw.geometry.values)
    raw = raw.to_crs(crs)

    def first_dataset(sources):
        """First dataset name in an Overture 'sources' value (list of {'dataset': ...})."""
        try:
            items = list(sources) if sources is not None else []
        except TypeError:
            items = []
        if not items:
            return "unknown"
        first = items[0]
        if isinstance(first, dict):
            return first.get("dataset") or "unknown"
        return getattr(first, "dataset", None) or "unknown"

    raw["area_m2"] = raw.geometry.area
    raw = raw[raw["area_m2"] >= MIN_AREA_PX * pixel_area_m2].reset_index(drop=True)

    buildings = gpd.GeoDataFrame(geometry=raw.geometry.values, crs=crs)
    buildings["area_m2"] = raw["area_m2"].values
    centres = buildings.geometry.centroid.to_crs(4326)
    buildings["lon"], buildings["lat"] = centres.x, centres.y
    source_col = raw["sources"] if "sources" in raw else [None] * len(raw)
    buildings["outline_source"] = ["overture_" + first_dataset(s) for s in source_col]
    buildings["new_candidate"] = False
    buildings["building_id"] = [f"B{i:05d}" for i in range(1, len(buildings) + 1)]
    buildings = buildings[["building_id", "area_m2", "lon", "lat", "outline_source", "new_candidate", "geometry"]]

    print(f"Buildings from Overture: {len(buildings)}")
    print()
    print("By outline_source:")
    print(buildings["outline_source"].value_counts().to_string())
    print()
    print("area_m2 summary:")
    print(buildings["area_m2"].describe().round(1).to_string())

## 4. Building type

**Adds:** `building_type`, `type_source`, `osm_name`.

Labels come from two free maps of places: **OpenStreetMap** and **Overture Places** (businesses and
services). Each label is attached to the nearest building within 15 m. Buildings with no label fall back
to a size rule: small = `residential`, large (over 400 m²) = `unknown_large`.

Limitation: only a few dozen buildings get a real label, so most types are guesses from size.

In [ ]:
# 4. Building type from OpenStreetMap and Overture Places, then a size rule for the rest.
import json, subprocess, sys, shutil, time
import requests

EDUCATION = {"school", "kindergarten", "college", "university"}
HEALTH = {"clinic", "hospital", "doctors", "pharmacy", "dentist", "health_post"}
COMMERCIAL_AMENITY = {"marketplace", "restaurant", "cafe", "bar", "fast_food", "bank", "fuel", "pub"}
INDUSTRIAL_BUILDING = {"industrial", "warehouse", "manufacture", "factory"}
COMMERCIAL_BUILDING = {"retail", "commercial", "supermarket", "kiosk", "office", "hotel"}
RELIGIOUS_BUILDING = {"church", "mosque", "temple", "chapel", "cathedral", "religious"}


def classify_tags(tags):
    """Turn OpenStreetMap tags into one of our building types (or None if not relevant)."""
    amenity, building = tags.get("amenity", ""), tags.get("building", "")
    if amenity in HEALTH or "healthcare" in tags or building in {"hospital", "clinic"}:
        return "health"
    if amenity in EDUCATION or building in EDUCATION:
        return "education"
    if amenity == "place_of_worship" or building in RELIGIOUS_BUILDING:
        return "religious"
    if building in INDUSTRIAL_BUILDING or "craft" in tags or tags.get("man_made") == "works":
        return "industrial"
    if "shop" in tags or "office" in tags or amenity in COMMERCIAL_AMENITY or building in COMMERCIAL_BUILDING:
        return "commercial"
    return None


OSM_SERVERS = ["https://overpass-api.de/api/interpreter",
               "https://overpass.kumi.systems/api/interpreter",
               "https://overpass.private.coffee/api/interpreter"]


def osm_places_with_backups(west, south, east, north):
    """Same OpenStreetMap question as above, but tries three servers, twice each."""
    box = f"({south},{west},{north},{east})"
    query = f"""[out:json][timeout:90];
(
  nwr["amenity"]{box};
  nwr["shop"]{box};
  nwr["office"]{box};
  nwr["craft"]{box};
  nwr["healthcare"]{box};
  nwr["man_made"="works"]{box};
  nwr["building"]["building"!~"^(yes|house|residential|detached|hut|shed|garage|roof|apartments)$"]{box};
);
out center tags;"""
    for attempt in range(2):
        for server in OSM_SERVERS:
            try:
                reply = requests.post(server, data={"data": query},
                                      headers={"User-Agent": "oseas-building-energy-notebook"}, timeout=120)
                reply.raise_for_status()
                rows = []
                for el in reply.json().get("elements", []):
                    kind = classify_tags(el.get("tags", {}))
                    lat = el.get("lat", el.get("center", {}).get("lat"))
                    lon = el.get("lon", el.get("center", {}).get("lon"))
                    if kind and lat is not None and lon is not None:
                        rows.append({"label_type": kind, "label_name": el["tags"].get("name"), "lon": lon, "lat": lat})
                print(f"OpenStreetMap: {len(rows)} labelled places (server: {server.split('/')[2]})")
                return pd.DataFrame(rows, columns=["label_type", "label_name", "lon", "lat"])
            except Exception as err:
                print(f"  OpenStreetMap server {server.split('/')[2]} failed: {str(err)[:80]}")
        time.sleep(10)
    print("OpenStreetMap: all servers failed, skipping it.")
    return pd.DataFrame(columns=["label_type", "label_name", "lon", "lat"])


def classify_overture_category(category):
    """Turn an Overture category (e.g. 'casual_eatery') into one of our building types."""
    c = category.lower() if isinstance(category, str) else ""
    if any(k in c for k in ["hospital", "clinic", "pharmacy", "dentist", "doctor", "medical", "health"]):
        return "health"
    if any(k in c for k in ["school", "college", "university", "kindergarten", "education", "campus", "learning"]):
        return "education"
    if any(k in c for k in ["church", "mosque", "temple", "religious", "worship", "cathedral"]):
        return "religious"
    if any(k in c for k in ["factory", "manufactur", "industrial", "warehouse", "mill", "workshop", "welding"]):
        return "industrial"
    if any(k in c for k in ["store", "shop", "market", "restaurant", "eatery", "food", "cafe", "bar", "bank",
                            "hotel", "lodging", "accommodation", "salon", "beauty", "office", "service",
                            "gas_station", "fuel", "supermarket", "bakery", "butcher", "retail", "atm",
                            "agency", "kiosk", "pub", "repair", "automotive", "dealer", "wholesale", "gym", "fitness"]):
        return "commercial"
    return None


def primary(value):
    """Overture stores some fields as {'primary': ...}; this pulls out the primary value.

    Works for plain strings, JSON strings, dicts, and struct-like scalars from parquet.
    """
    if isinstance(value, str):
        if value.startswith("{"):
            try:
                value = json.loads(value)
            except ValueError:
                return value
        else:
            return value
    if isinstance(value, dict):
        return value.get("primary")
    if hasattr(value, "get"):            # mapping-like struct returned from parquet
        try:
            return value.get("primary")
        except Exception:
            return value
    return getattr(value, "primary", value)   # struct scalar exposing a .primary attribute



def overture_places(west, south, east, north):
    """Load Overture Places for the box (local parquet if present, else download) and type each place."""
    raw = None
    if os.path.exists(OVERTURE_PLACES):
        try:
            raw = gpd.read_parquet(OVERTURE_PLACES)
            print(f"Overture Places: read {len(raw)} rows from {OVERTURE_PLACES}")
        except Exception as err:
            print("Overture Places: could not read local file, will download instead. Reason:", str(err)[:120])
            raw = None
    if raw is None:
        out_file = os.path.join(WORK_DIR, "overture_places.geojson")
        overture_cmd = shutil.which("overturemaps")
        if overture_cmd is None:
            exe = "overturemaps.exe" if os.name == "nt" else "overturemaps"
            overture_cmd = os.path.join(os.path.dirname(sys.executable), exe)
        try:
            subprocess.run([overture_cmd, "download", f"--bbox={west},{south},{east},{north}",
                            "-f", "geojson", "--type=place", "-o", out_file], check=True, timeout=900)
            raw = gpd.read_file(out_file)
        except Exception as err:
            print("Overture Places: download failed, skipping it. Reason:", str(err)[:120])
            return pd.DataFrame(columns=["label_type", "label_name", "lon", "lat"])
    if "basic_category" in raw:
        raw["category"] = raw["basic_category"]
    elif "taxonomy" in raw:
        raw["category"] = raw["taxonomy"].apply(primary)
    else:
        raw["category"] = raw["categories"].apply(primary)
    raw["label_name"] = raw["names"].apply(primary)
    raw = raw[raw["confidence"].fillna(0) >= PLACES_MIN_CONFIDENCE].copy()
    raw["label_type"] = raw["category"].apply(classify_overture_category)

    print("Overture categories in this area, and the type we give each:")
    print(raw.groupby(["category", "label_type"], dropna=False).size().rename("places").to_string())
    print()

    ovt = pd.DataFrame({"label_type": raw["label_type"], "label_name": raw["label_name"],
                        "lon": raw.geometry.x, "lat": raw.geometry.y})
    return ovt[ovt["label_type"].notna()]


# ---- Get labels from both sources -------------------------------------------
west, south, east, north = buildings.to_crs(4326).total_bounds
osm = osm_places_with_backups(west, south, east, north).assign(label_source="osm")
ovt_places = overture_places(west, south, east, north).assign(label_source="overture")
print(f"Overture places we can use: {len(ovt_places)}   |   OpenStreetMap places: {len(osm)}")

labels_all = pd.concat([osm, ovt_places], ignore_index=True)
labels_all = gpd.GeoDataFrame(labels_all, geometry=gpd.points_from_xy(labels_all["lon"], labels_all["lat"]),
                              crs=4326).to_crs(crs)

buildings["building_type"] = None
buildings["osm_name"] = None
buildings["type_source"] = "size_rule"

if len(labels_all) > 0:
    matched = gpd.sjoin_nearest(labels_all[["label_type", "label_name", "label_source", "geometry"]],
                                buildings[["building_id", "geometry"]],
                                max_distance=POI_MAX_DISTANCE_M, distance_col="dist_m")
    matched = matched.dropna(subset=["building_id"]).sort_values("dist_m")
    best = matched.drop_duplicates("building_id").set_index("building_id")
    hit = buildings["building_id"].isin(best.index)
    ids = buildings.loc[hit, "building_id"]
    buildings.loc[hit, "building_type"] = ids.map(best["label_type"])
    buildings.loc[hit, "osm_name"] = ids.map(best["label_name"])
    buildings.loc[hit, "type_source"] = ids.map(best["label_source"])
    print(f"Labels within {POI_MAX_DISTANCE_M} m of a building: {len(matched)} of {len(labels_all)}")

untagged = buildings["building_type"].isna()
buildings.loc[untagged & (buildings["area_m2"] >= LARGE_BUILDING_M2), "building_type"] = "unknown_large"
buildings.loc[untagged & (buildings["area_m2"] < LARGE_BUILDING_M2), "building_type"] = "residential"

print()
print(buildings.groupby(["building_type", "type_source"]).size().rename("buildings"))

## 4b. Road access

**Adds:** `dist_road_m`, `dist_main_road_m`, `dist_paved_road_m` (metres).

Why: in Kenya, better-off households tend to live closer to good roads, and roads bring electricity lines
and customers. Only about 2% of roads here have their surface (paved or not) recorded in OpenStreetMap,
so we use **distance to a main road** (primary, secondary, tertiary) instead of "paved road".

Roads come from Overture's transportation layer (based on OpenStreetMap); the Overpass API is used only if that file is missing.

In [ ]:
# 4b. Road access: distance from every building to the nearest road, main road and paved road.
from shapely.geometry import LineString

MAIN_ROADS = {"motorway", "trunk", "primary", "secondary", "tertiary",
              "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link"}
PAVED = {"asphalt", "paved", "concrete", "concrete:plates", "concrete:lanes", "paving_stones", "sett", "chipseal"}
NOT_ROADS = {"footway", "path", "steps", "cycleway", "bridleway", "corridor", "pedestrian",
             "construction", "proposed", "platform", "elevator"}


def osm_roads(west, south, east, north, pad=0.01):
    """Download all roads in the box (plus about 1 km around it) from OpenStreetMap."""
    box = f"({south - pad},{west - pad},{north + pad},{east + pad})"
    query = f'[out:json][timeout:120];way["highway"]{box};out tags geom;'
    for attempt in range(2):
        for server in OSM_SERVERS:
            try:
                reply = requests.post(server, data={"data": query},
                                      headers={"User-Agent": "oseas-building-energy-notebook"}, timeout=180)
                reply.raise_for_status()
                rows = []
                for el in reply.json().get("elements", []):
                    tags, pts = el.get("tags", {}), el.get("geometry", [])
                    if tags.get("highway", "") in NOT_ROADS or len(pts) < 2:
                        continue
                    rows.append({"highway": tags.get("highway"), "surface": tags.get("surface"),
                                 "road_name": tags.get("name"),
                                 "geometry": LineString([(p["lon"], p["lat"]) for p in pts])})
                print(f"OpenStreetMap roads: {len(rows)} road pieces (server: {server.split('/')[2]})")
                return gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else None
            except Exception as err:
                print(f"  OpenStreetMap server {server.split('/')[2]} failed: {str(err)[:80]}")
        time.sleep(10)
    return None


def overture_roads(path):
    """Road lines from an Overture transportation 'segments' geoparquet file."""
    seg = gpd.read_parquet(path)
    seg = seg[seg["subtype"] == "road"].copy()

    def first_surface(rs):
        try:
            items = list(rs) if rs is not None else []
        except TypeError:
            items = []
        for it in items:
            v = it.get("value") if isinstance(it, dict) else getattr(it, "value", None)
            if v:
                return v
        return None

    def road_name(nm):
        if isinstance(nm, dict):
            return nm.get("primary")
        return getattr(nm, "primary", None)

    seg["highway"] = seg["class"]
    seg["surface"] = seg["road_surface"].apply(first_surface) if "road_surface" in seg else None
    seg["road_name"] = seg["names"].apply(road_name) if "names" in seg else None
    seg = seg[~seg["highway"].isin(NOT_ROADS)].copy()          # drop footway/path/steps/cycleway/etc.
    roads = gpd.GeoDataFrame(seg[["highway", "surface", "road_name", "geometry"]].reset_index(drop=True),
                             geometry="geometry", crs=seg.crs)
    print(f"Overture roads: {len(roads)} road pieces (from {os.path.basename(path)})")
    return roads


def distance_to(lines):
    """Distance in metres from each building to the nearest line in `lines`."""
    if lines is None or len(lines) == 0:
        return pd.Series(np.nan, index=buildings.index)
    joined = gpd.sjoin_nearest(buildings[["geometry"]], lines[["geometry"]], distance_col="d")
    return joined.groupby(level=0)["d"].min()


west, south, east, north = buildings.to_crs(4326).total_bounds
OVERTURE_SEGMENTS = os.path.join(BASE, "overture_download", "segments.parquet")
if os.path.exists(OVERTURE_SEGMENTS):
    roads = overture_roads(OVERTURE_SEGMENTS)
    print("Roads source: Overture transportation segments file")
else:
    roads = osm_roads(west, south, east, north)
    print("Roads source: Overpass (live OpenStreetMap download)")

if roads is None:
    print("Could not get roads; the distance columns are left empty. Re-run this cell later.")
    for col in ["dist_road_m", "dist_main_road_m", "dist_paved_road_m"]:
        buildings[col] = np.nan
else:
    roads = roads.to_crs(crs)
    roads["is_main"] = roads["highway"].isin(MAIN_ROADS)
    roads["is_paved"] = roads["surface"].isin(PAVED)
    buildings["dist_road_m"] = distance_to(roads)
    buildings["dist_main_road_m"] = distance_to(roads[roads["is_main"]])
    buildings["dist_paved_road_m"] = distance_to(roads[roads["is_paved"]])

    print(f"  main roads: {int(roads['is_main'].sum())}   other roads: {int((~roads['is_main']).sum())}")
    print(f"  roads with a surface recorded: {roads['surface'].notna().mean():.0%}"
          f"   (paved: {int(roads['is_paved'].sum())})")
    print()
    for col, label in [("dist_road_m", "any road"), ("dist_main_road_m", "a main road"),
                       ("dist_paved_road_m", "a paved road")]:
        d = buildings[col]
        print(f"Distance to {label:13s}: median {d.median():6.0f} m | within 100 m: {(d <= 100).mean():4.0%}"
              f" | within 500 m: {(d <= 500).mean():4.0%}")

    # Map: buildings coloured by distance to a main road (darker = closer); main roads in red
    road_map = buildings[["building_id", "area_m2", "dist_main_road_m", "geometry"]].to_crs(4326).explore(
        column="dist_main_road_m", cmap="viridis_r", tiles="Esri.WorldImagery",
        style_kwds={"weight": 0.5, "fillOpacity": 0.7}, name="buildings", legend=True)
    roads[roads["is_main"]].to_crs(4326).explore(m=road_map, color="red", style_kwds={"weight": 3},
                                                 name="main roads")
    os.makedirs(WORK_DIR, exist_ok=True)
    road_map.save(os.path.join(WORK_DIR, "road_access_map.html"))
    display(road_map)

## 4c. Outbuildings

**Adds:** `main_building_id`, and the type `outbuilding`.

Eliane says homes are usually above 50–80 m² (a Kenyan study uses 120 m² as a typical single-family
roof). Many small roofs are kitchens, toilets or stores next to the main house. A small building
(under 50 m²) within 15 m of a bigger home becomes that home's outbuilding. A small building with
no bigger home nearby stays a home.

In [ ]:
# 4c. Outbuildings: small buildings next to a bigger home are part of that home, not separate households.

res = buildings["building_type"] == "residential"
small = res & (buildings["area_m2"] < OUTBUILDING_MAX_M2)
big = res & ~small
buildings["main_building_id"] = None

homes_big = buildings.loc[big, ["building_id", "geometry"]].rename(columns={"building_id": "main_id"})
near = gpd.sjoin_nearest(buildings.loc[small, ["geometry"]], homes_big,
                         max_distance=ATTACH_DISTANCE_M, distance_col="gap_m").sort_values("gap_m")
near = near[~near.index.duplicated()]
buildings.loc[near.index, "building_type"] = "outbuilding"
buildings.loc[near.index, "main_building_id"] = near["main_id"].values

print(f"Small homes (under {OUTBUILDING_MAX_M2} m2): {int(small.sum())}")
print(f"  -> now outbuildings of a bigger home within {ATTACH_DISTANCE_M} m: {len(near)}")
print(f"  -> still homes (no bigger home nearby): {int(small.sum()) - len(near)}")

## 4d. Buildings to check by hand

Large buildings with no map label are listed with a Google Maps link, so a person can look at the satellite
view and write the real type into `manual_labels.csv`. The notebook then reads that file and uses those
answers, matching each one to the nearest building by location.

In [ ]:
# 4d. Buildings to check by hand: list large unlabelled / industrial buildings with a Google Maps link,
#     then apply any manual corrections from manual_labels.csv (matched by location, not building_id).
import os
import folium

os.makedirs(WORK_DIR, exist_ok=True)

# ---- A. Review list: large buildings a person should look at ----------------
to_check = buildings[buildings["building_type"].isin(["unknown_large", "industrial"])].copy()
to_check = to_check.sort_values("area_m2", ascending=False).reset_index(drop=True)
to_check["rank"] = np.arange(1, len(to_check) + 1)
to_check["lat"] = to_check["lat"].round(6)
to_check["lon"] = to_check["lon"].round(6)
to_check["google_maps"] = [f"https://www.google.com/maps/@{lat},{lon},19z/data=!3m1!1e3"
                           for lat, lon in zip(to_check["lat"], to_check["lon"])]

review = to_check[["rank", "building_id", "building_type", "area_m2", "lat", "lon", "google_maps"]].copy()
review["area_m2"] = review["area_m2"].round(0)
review.to_csv(os.path.join(WORK_DIR, "buildings_to_check.csv"), index=False)
print(f"{len(review)} buildings to check by hand. Top 20:")
print(review.head(20).to_string(index=False))

# ---- A.2 Map of just these buildings, Esri satellite ------------------------
if len(to_check) > 0:
    rev4326 = to_check.to_crs(4326)
    cen = to_check.geometry.centroid.to_crs(4326)
    fmap = folium.Map(location=[cen.y.mean(), cen.x.mean()], zoom_start=15, tiles=None)
    folium.TileLayer("Esri.WorldImagery", name="satellite").add_to(fmap)
    for (_, r), g in zip(rev4326.iterrows(), rev4326.geometry):
        popup_html = (f"<b>rank {int(r['rank'])}</b> &mdash; {r['building_type']}, {r['area_m2']:.0f} m2<br>"
                      f'<a href="{r["google_maps"]}" target="_blank">Open in Google Maps (satellite)</a>')
        folium.GeoJson(g.__geo_interface__,
                       tooltip=f"rank {int(r['rank'])}, {r['area_m2']:.0f} m2",
                       popup=folium.Popup(popup_html, max_width=300),
                       style_function=lambda f: {"color": "#e4572e", "weight": 2, "fillOpacity": 0.3}).add_to(fmap)
    fmap.save(os.path.join(WORK_DIR, "buildings_to_check.html"))
    print("Saved:", os.path.join(WORK_DIR, "buildings_to_check.csv"),
          "and", os.path.join(WORK_DIR, "buildings_to_check.html"))

# ---- A.3 Create an empty manual_labels.csv template if it does not exist -----
MANUAL_LABELS = os.path.join(BASE, "manual_labels.csv")
if not os.path.exists(MANUAL_LABELS):
    pd.DataFrame(columns=["lat", "lon", "building_type", "note"]).to_csv(MANUAL_LABELS, index=False)
    print("Created empty template:", MANUAL_LABELS)

# ---- B. Apply manual corrections (match the nearest building within 15 m) ----
ALLOWED_TYPES = {"residential", "commercial", "education", "health", "religious",
                 "industrial", "agriculture", "unknown_large"}
manual = pd.read_csv(MANUAL_LABELS)
applied, unmatched, rejected = 0, [], []
if len(manual) > 0:
    pts = gpd.GeoDataFrame(manual.copy(), geometry=gpd.points_from_xy(manual["lon"], manual["lat"]),
                           crs=4326).to_crs(crs)
    for _, prow in pts.iterrows():
        bt = str(prow["building_type"]).strip()
        if bt not in ALLOWED_TYPES:
            rejected.append((prow["lat"], prow["lon"], bt))
            continue
        dists = buildings.geometry.distance(prow.geometry)
        j = dists.idxmin()
        if dists.loc[j] <= 15:
            buildings.loc[j, "building_type"] = bt
            buildings.loc[j, "type_source"] = "manual"
            applied += 1
        else:
            unmatched.append((prow["lat"], prow["lon"]))

print(f"Manual labels applied: {applied}")
if rejected:
    print("Rejected rows (building_type not allowed):")
    for lat, lon, bt in rejected:
        print(f"  ({lat}, {lon}): '{bt}' is not one of {sorted(ALLOWED_TYPES)}")
if unmatched:
    print("Rows that matched no building within 15 m:")
    for lat, lon in unmatched:
        print(f"  ({lat}, {lon})")

print()
print("building_type / type_source after 4d:")
print(buildings.groupby(["building_type", "type_source"]).size().rename("buildings").to_string())

## 5. Homes' electricity

**Adds:** `household_id`, `buildings_in_household`, `annual_kwh`, `peak_kw`, `road_factor`,
`load_pattern`, `peak_hour`, `evening_share` for homes.

Eliane's method, using household load profiles simulated with RAMP from a household survey in Nigeria (project dataset):
1. Find the 4 typical daily patterns in the readings.
2. Group homes that are close together into one household (compound).
3. Give each household a yearly total: bigger roofs use more, **closer to a main road uses more**
   (`road_factor`, new in this version), plus some random variation. The average stays at 62.4 kWh a month.
4. Give each household one of the 4 patterns, and find its peak.
5. Split each household's numbers between its buildings by roof size.

In [ ]:
# 5. Electricity for homes: households, yearly kWh, daily pattern and peak.
from sklearn.cluster import DBSCAN, KMeans

rng = np.random.default_rng(SEED)

# ---- a) Typical daily patterns from survey-based household profiles (Nigeria, RAMP) ----
readings = pd.read_csv(PROFILE_CSV).set_index("Hour")
hh_cols = [c for c in readings.columns if c.startswith("HH_")]
hours = readings.index.to_numpy()
X = readings[hh_cols].to_numpy(dtype=float).T          # one row per surveyed household, 24 columns
X = X[~np.isnan(X).any(axis=1)]
X = X[X.sum(axis=1) > 0]
X_norm = X / X.sum(axis=1, keepdims=True)              # each row now adds up to 1: shape only, not size

kmeans = KMeans(n_clusters=N_ARCHETYPES, random_state=42, n_init=20)
labels = kmeans.fit_predict(X_norm)
shares = np.bincount(labels, minlength=N_ARCHETYPES) / len(labels)
print(f"Survey-based household profiles used (Nigeria, RAMP): {len(X_norm)}; share in each pattern: {np.round(shares, 2)}")

# ---- b) Group homes into households (compounds) ---------------------------
homes = buildings[buildings["building_type"] == "residential"].copy()
xy = np.column_stack([homes.geometry.centroid.x, homes.geometry.centroid.y])
homes["household_id"] = DBSCAN(eps=COMPOUND_DISTANCE_PX * pixel_m, min_samples=1).fit_predict(xy) + 1

households = homes.groupby("household_id").agg(roof_area_m2=("area_m2", "sum"),
                                               n_buildings=("building_id", "size"))

# ---- c) Monthly use per household: bigger roof area means more use --------
area_used = households["roof_area_m2"].clip(upper=households["roof_area_m2"].quantile(ROOF_CLIP_QUANTILE))

# Road effect (from section 4b): households closer to a main road use more electricity.
# Rank every household by its distance to a main road: closest gets +ROAD_EFFECT, farthest gets -ROAD_EFFECT.
households["dist_main_road_m"] = homes.groupby("household_id")["dist_main_road_m"].min()
rank = households["dist_main_road_m"].rank(pct=True)                 # 0 = closest, 1 = farthest
households["road_factor"] = (1 + ROAD_EFFECT * (1 - 2 * rank)).fillna(1.0)   # 1.0 if road data is missing

weight = (area_used ** ALPHA
          * (1.0 + BUILDING_WEIGHT * (households["n_buildings"] - 1))
          * households["road_factor"]
          * rng.lognormal(mean=0.0, sigma=SIGMA_HOUSEHOLD, size=len(households)))
households["monthly_kwh"] = MONTHLY_KWH_MEAN * weight / weight.mean()   # average stays at MONTHLY_KWH_MEAN
households["annual_kwh"] = households["monthly_kwh"] * 12
daily_kwh = households["monthly_kwh"] / 30.0

# ---- d) Daily pattern and peak for each household -------------------------
households["load_pattern"] = rng.choice(N_ARCHETYPES, size=len(households), p=shares)
members = {k: np.where(labels == k)[0] for k in range(N_ARCHETYPES)}
evening = (hours >= 18) & (hours <= 22)

peak_kw, peak_hour, evening_share = [], [], []
for pattern, kwh_day in zip(households["load_pattern"], daily_kwh):
    days = X_norm[rng.choice(members[pattern], size=DAYS_SAMPLED)]   # surveyed days of this pattern
    average_day = days.mean(axis=0)
    peak_kw.append(kwh_day * days.max())        # highest hourly use over the sampled days (kWh in 1 hour = kW)
    peak_hour.append(hours[average_day.argmax()])
    evening_share.append(average_day[evening].sum())
households["peak_kw"] = peak_kw
households["peak_hour"] = peak_hour
households["evening_share"] = evening_share

# ---- e) Share each household's numbers between its buildings by roof size --
homes = homes.join(households.add_prefix("hh_"), on="household_id")
roof_share = homes["area_m2"] / homes["hh_roof_area_m2"]
homes["annual_kwh"] = homes["hh_annual_kwh"] * roof_share
homes["peak_kw"] = homes["hh_peak_kw"] * roof_share
homes["load_pattern"] = homes["hh_load_pattern"]
homes["peak_hour"] = homes["hh_peak_hour"]
homes["evening_share"] = homes["hh_evening_share"]
homes["buildings_in_household"] = homes["hh_n_buildings"]
homes["road_factor"] = homes["hh_road_factor"]

new_cols = ["household_id", "buildings_in_household", "annual_kwh", "peak_kw", "road_factor",
            "load_pattern", "peak_hour", "evening_share"]
buildings = buildings.drop(columns=[c for c in new_cols if c in buildings.columns])
buildings = buildings.merge(homes[["building_id"] + new_cols], on="building_id", how="left")
for col in ["household_id", "buildings_in_household", "load_pattern", "peak_hour"]:
    buildings[col] = buildings[col].astype("Int64")      # whole numbers, empty where not a home

# ---- f) Checks -------------------------------------------------------------
total = buildings["annual_kwh"].sum()
expected = len(households) * MONTHLY_KWH_MEAN * 12
assert abs(total - expected) < 1e-6 * expected, "Building totals do not add up to the household total."
print(f"Homes: {len(homes)}  ->  households: {len(households)}")
print("Largest household:", int(households["n_buildings"].max()), "buildings (a very large number means dense housing got chained together)")
print(f"Average household use: {households['annual_kwh'].mean():.0f} kWh a year")
near, far = rank <= 0.25, rank >= 0.75
print(f"  closest 25% to a main road:  {households.loc[near, 'annual_kwh'].mean():.0f} kWh a year")
print(f"  farthest 25% from a main road: {households.loc[far, 'annual_kwh'].mean():.0f} kWh a year")
print(f"Total for the area: {total / 1000:.0f} MWh a year")
print("Buildings left without numbers (outbuildings, schools, clinics, shops...):", int(buildings['annual_kwh'].isna().sum()))

### 5b. Outbuildings join their household

**Adds:** outbuildings get their main home's `household_id`, and 0 kWh (their use is counted in the main home).

In [ ]:
# 5b. Outbuildings join their main home's household and use no electricity of their own.
is_out = buildings["building_type"] == "outbuilding"
by_id = buildings.set_index("building_id")
main = buildings.loc[is_out, "main_building_id"]
buildings.loc[is_out, "household_id"] = main.map(by_id["household_id"]).values
buildings.loc[is_out, "buildings_in_household"] = main.map(by_id["buildings_in_household"]).values
buildings.loc[is_out, "road_factor"] = main.map(by_id["road_factor"]).values
buildings.loc[is_out, ["annual_kwh", "peak_kw"]] = 0.0
print(f"Outbuildings linked to a household (0 kWh each): {int(is_out.sum())}")
print("Buildings still without numbers (schools, clinics, shops, large unknown):", int(buildings["annual_kwh"].isna().sum()))

## 6. Other buildings' electricity with RAMP

**Adds:** `annual_kwh`, `peak_kw`, `peak_hour`, `evening_share`, `size_scale`, `profile_source`
for schools, clinics, churches, shops, workshops and large unlabelled buildings.

The household load profiles (simulated with RAMP from a household survey in Nigeria, project dataset) are for homes only. For other buildings we use **RAMP**: you describe a typical
building's appliances (how many, how many watts, how long and at what times of day they are used), and
RAMP simulates realistic minute-by-minute use for many days, with natural randomness.

- **6a** holds the appliance lists. **They are placeholders**: ask the experts on Monday to correct them.
  A published set of measured RAMP profiles for schools and health facilities in sub-Saharan Africa
  (Stevanato et al., see references) could replace them later.
- **6b** runs RAMP and gives each building its numbers, scaled by its size compared with the typical building.

In [ ]:
# 6a. Appliance lists for buildings that are not homes. PLACEHOLDERS: confirm with Eliane and the experts.
#
# Each building type describes ONE typical building of `ref_area_m2` square metres.
# Each appliance line is: (name, how many, watts each, minutes used per day, hours when it can be on)
#   - "hours when it can be on" is a list of (start hour, end hour), up to 3 of them, e.g. [(8, 12), (13, 17)]
#   - write "always" instead for a fridge: it runs all day, switching its motor on and off
# `weekdays_only=True` means the building is closed on Saturday and Sunday (fridges still run).
#
# Bigger or smaller buildings of the same type are scaled by size (see SIZE_EXPONENT in the next cell).
#
# Timing pattern: households peak in the evening (lighting); businesses and mills use power mainly in the
# daytime. Source: NREL (2018), Productive Use of Energy in African Micro-Grids, NREL/TP-7A40-71663.

APPLIANCES = {
    "education": {"ref_area_m2": 300, "weekdays_only": True, "items": [      # small primary school
        ("lights",          10,   20, 240, [(8, 12), (13, 17)]),
        ("computers",        4,   60, 240, [(8, 17)]),
        ("phone_charging",   5,    8, 120, [(8, 17)]),
        ("fridge",           1,  100, None, "always"),
    ]},
    "health": {"ref_area_m2": 150, "weekdays_only": False, "items": [        # dispensary / small clinic
        ("lights",           8,   20, 360, [(6, 8), (18, 24), (0, 6)]),
        ("vaccine_fridge",   1,   60, None, "always"),
        ("computer",         1,   60, 300, [(8, 17)]),
        ("sterilizer",       1, 1500,  30, [(8, 16)]),
        ("fans",             2,   50, 240, [(11, 17)]),
        ("phone_charging",   3,    8, 120, [(8, 20)]),
    ]},
    "religious": {"ref_area_m2": 250, "weekdays_only": False, "items": [     # church or mosque
        ("lights",          15,   20, 120, [(5, 7), (18, 21)]),
        ("sound_system",     1,  300,  90, [(6, 13), (17, 21)]),
        ("fans",             2,   50,  90, [(10, 16)]),
    ]},
    "commercial": {"ref_area_m2": 60, "weekdays_only": False, "items": [     # shop, kiosk, bar, salon: busy by day
        ("lights",            4,   20, 240, [(7, 9), (17, 20)]),
        ("drinks_fridge",     1,  150, None, "always"),
        ("tv_or_radio",       1,   50, 540, [(8, 18)]),
        ("phone_charging",    4,    8, 300, [(8, 18)]),
        ("tools_or_machines", 1,  300, 240, [(9, 13), (14, 18)]),
    ]},
    "industrial": {"ref_area_m2": 150, "weekdays_only": False, "items": [    # workshop, posho (maize) mill, welding
        ("lights",           4,   20, 120, [(6, 8), (17, 20)]),
        ("motor_or_mill",    1, 3000, 180, [(8, 17)]),
        ("welder",           1, 4000,  45, [(9, 16)]),
        ("power_tools",      2,  500,  60, [(8, 17)]),
    ]},
    "agriculture": {"ref_area_m2": 1000, "weekdays_only": False, "items": [  # greenhouse / farm buildings: PLACEHOLDER
        ("irrigation_pump",      1, 1500, 120, [(6, 10), (15, 18)]),
        ("lights",               6,   20, 120, [(5, 7), (18, 20)]),
        ("fans_or_ventilation",  2,  200, 360, [(10, 17)]),
    ]},
    "unknown_large": {"ref_area_m2": 400, "weekdays_only": False, "items": [  # large building with no label: GUESS
        ("lights",          12,   20, 300, [(6, 8), (18, 23)]),
        ("fridge",           1,  150, None, "always"),
        ("tv",               2,   60, 240, [(18, 23)]),
        ("sockets",          4,   60, 180, [(8, 22)]),
    ]},
}
print("Building types with an appliance list:", ", ".join(APPLIANCES))

In [ ]:
# 6b. RAMP: simulate a minute-by-minute electricity profile for each non-home building type,
#     then give every non-home building its yearly kWh, peak kW and daily pattern.
import random
from ramp import User, UseCase



def make_ramp_user(kind, spec):
    """Turn one appliance list from 6a into a RAMP 'User' (RAMP's word for one type of building)."""
    user = User(user_name=kind, num_users=1)
    day_type = 0 if spec.get("weekdays_only") else 2          # RAMP: 0 = weekdays only, 2 = every day
    for name, number, watts, minutes, hours in spec["items"]:
        if hours == "always":                                   # fridge: on all day, motor cycles on/off
            a = user.add_appliance(number=number, power=watts, num_windows=1, func_time=1440,
                                   func_cycle=30, fixed="yes", fixed_cycle=1, name=name)
            a.windows(window_1=[0, 1440])
            a.specific_cycle_1(p_11=watts, t_11=20, p_12=5, t_12=10)
            a.cycle_behaviour(cw11=[0, 1440])
            continue
        windows = [[int(start * 60), int(end * 60)] for start, end in hours]
        open_minutes = sum(end - start for start, end in windows)
        assert minutes <= open_minutes, f"{kind}/{name}: {minutes} minutes do not fit in the hours given"
        a = user.add_appliance(number=number, power=watts, num_windows=len(windows), func_time=minutes,
                               time_fraction_random_variability=0.2, func_cycle=min(30, minutes),
                               wd_we_type=day_type, name=name)
        a.windows(**{f"window_{i + 1}": w for i, w in enumerate(windows)}, random_var_w=0.1)
    return user


end = (pd.Timestamp(RAMP_START) + pd.Timedelta(days=RAMP_DAYS - 1)).strftime("%Y-%m-%d")
ramp_summary, ramp_hourly_kw = {}, {}
for kind, spec in APPLIANCES.items():
    random.seed(SEED)                                           # same results every run
    np.random.seed(SEED)
    case = UseCase(users=[make_ramp_user(kind, spec)], date_start=RAMP_START, date_end=end)
    minutes_w = np.asarray(case.generate_daily_load_profiles(), dtype=float).reshape(-1, 1440)  # watts, 1 row per day
    hourly_kw = minutes_w.reshape(len(minutes_w), 24, 60).mean(axis=2) / 1000   # average kW in each hour, per day
    average_day = hourly_kw.mean(axis=0)
    ramp_hourly_kw[kind] = average_day                          # used for the area load curve in section 9
    ramp_summary[kind] = {"ref_area_m2": spec["ref_area_m2"],
                          "daily_kwh": hourly_kw.sum(axis=1).mean(),
                          "peak_kw": hourly_kw.max(),           # highest hourly use (same as for homes)
                          "peak_hour": int(average_day.argmax()),
                          "evening_share": average_day[18:23].sum() / average_day.sum()}
ramp_summary = pd.DataFrame(ramp_summary).T
print("One typical building of each type (from RAMP):")
print(ramp_summary.astype(float).round(2).to_string())
print()

# ---- Give each non-home building its numbers, scaled by its size ------------
is_ramp = buildings["building_type"].isin(list(APPLIANCES))
kind = buildings.loc[is_ramp, "building_type"]
ref = kind.map(ramp_summary["ref_area_m2"]).astype(float)
size_scale = ((buildings.loc[is_ramp, "area_m2"] / ref) ** SIZE_EXPONENT).clip(*SCALE_LIMITS)
buildings["size_scale"] = np.nan
buildings.loc[is_ramp, "size_scale"] = size_scale
buildings.loc[is_ramp, "annual_kwh"] = kind.map(ramp_summary["daily_kwh"]).astype(float) * 365 * size_scale
buildings.loc[is_ramp, "peak_kw"] = kind.map(ramp_summary["peak_kw"]).astype(float) * size_scale
buildings.loc[is_ramp, "peak_hour"] = kind.map(ramp_summary["peak_hour"]).astype(int).values
buildings.loc[is_ramp, "evening_share"] = kind.map(ramp_summary["evening_share"]).astype(float)

buildings["profile_source"] = np.select(
    [buildings["building_type"] == "residential", buildings["building_type"] == "outbuilding", is_ramp],
    ["household_readings", "part_of_household", "ramp_" + buildings["building_type"].astype(str)], default="none")

print(buildings[is_ramp].groupby("building_type").agg(
    buildings=("building_id", "size"), median_m2=("area_m2", "median"),
    avg_kwh_year=("annual_kwh", "mean"), total_mwh_year=("annual_kwh", lambda s: s.sum() / 1000)).round(1).to_string())
print()
print("Buildings still without numbers:", int(buildings["annual_kwh"].isna().sum()))

## 7. Weather and cooling demand

**Adds:** `cooling_category` (Low / Medium / High).

Downloads one year of daily temperatures for the centre of the area from **NASA POWER** (free) and
counts **cooling degree days**: for each day, how many degrees the average temperature is above 18 °C,
added up over the year. More degree days = more need for fans, fridges and air conditioning.

The area is small, so every building gets the same weather. Shops and clinics go one level higher,
because they keep drinks, food or medicine cold.

In [ ]:
# 7. Weather: how hot the area gets (NASA POWER, free), and a cooling category for every building.
lon_c, lat_c = buildings.to_crs(4326).total_bounds.reshape(2, 2).mean(axis=0)
params = {"parameters": "T2M,T2M_MAX", "community": "RE", "longitude": round(lon_c, 4),
          "latitude": round(lat_c, 4), "start": f"{WEATHER_YEAR}0101", "end": f"{WEATHER_YEAR}1231", "format": "JSON"}
LEVELS = ["Low", "Medium", "High"]
daily_temp = None      # filled below; section 9 uses it for the monthly profile

try:
    reply = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", params=params, timeout=120)
    reply.raise_for_status()
    p = reply.json()["properties"]["parameter"]
    t_mean = pd.Series(p["T2M"], dtype=float).replace(-999, np.nan)        # daily average temperature, deg C
    t_max = pd.Series(p["T2M_MAX"], dtype=float).replace(-999, np.nan)     # daily maximum temperature, deg C
    t_mean.index = pd.to_datetime(t_mean.index, format="%Y%m%d")
    t_max.index = t_mean.index
    daily_temp = t_mean
    cdd = (t_mean - 18).clip(lower=0).sum()                                # cooling degree days, base 18 deg C
    site = "Low" if cdd < COOLING_MEDIUM_FROM else ("Medium" if cdd < COOLING_HIGH_FROM else "High")
    print(f"{WEATHER_YEAR} at this site: average {t_mean.mean():.1f} C, hottest day {t_max.max():.1f} C, "
          f"days above 30 C: {int((t_max >= 30).sum())}")
    print(f"Cooling degree days (base 18 C): {cdd:.0f}  ->  site cooling need: {site}")
except Exception as err:
    site = None
    print("Could not reach NASA POWER, cooling category left empty. Reason:", str(err)[:100])

buildings["cooling_category"] = site
if site:
    # shops, bars and clinics keep drinks, food or medicine cold, so one level higher
    up = buildings["building_type"].isin(["commercial", "health"])
    buildings.loc[up, "cooling_category"] = LEVELS[min(LEVELS.index(site) + 1, 2)]
    print(buildings["cooling_category"].value_counts().to_string())

## 7b. Hotter-climate what-if (Mombasa)

Our site's temperatures vary only a little (monthly average about 18-21 C), so we also show the same
buildings under a hotter Kenyan climate (Mombasa) as a clearly labelled what-if. The temperature effect on
the peak uses the placeholder `TEMP_SENSITIVITY`, so this is an assumption to confirm, not a measurement.

In [ ]:
# 7b. Hotter-climate what-if: the same buildings under Mombasa temperatures (clearly labelled as an assumption).
import os
os.makedirs(WORK_DIR, exist_ok=True)

# Area peak from the average day, reconstructed here (section 9 recomputes the same thing).
_order = np.argsort(np.asarray(hours) % 24)
_ps = kmeans.cluster_centers_[:, _order]
_ps = _ps / _ps.sum(axis=1, keepdims=True)
_hb = np.zeros((len(buildings), 24))
_home = (buildings["profile_source"] == "household_readings").to_numpy()
_hb[_home] = (_ps[buildings.loc[_home, "load_pattern"].astype(int).to_numpy()]
              * (buildings.loc[_home, "annual_kwh"].to_numpy() / 365)[:, None])
for _k in APPLIANCES:
    _m = (buildings["building_type"] == _k).to_numpy()
    if _m.any():
        _hb[_m] = ramp_hourly_kw[_k][None, :] * buildings.loc[_m, "size_scale"].to_numpy()[:, None]
area_peak_kw = float(_hb.sum(axis=0).max())


def _nasa_daily(lat, lon):
    """Download one year of daily T2M and T2M_MAX from NASA POWER (same call as section 7)."""
    pr = {"parameters": "T2M,T2M_MAX", "community": "RE", "longitude": round(lon, 4),
          "latitude": round(lat, 4), "start": f"{WEATHER_YEAR}0101", "end": f"{WEATHER_YEAR}1231", "format": "JSON"}
    r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", params=pr, timeout=120)
    r.raise_for_status()
    p = r.json()["properties"]["parameter"]
    tm = pd.Series(p["T2M"], dtype=float).replace(-999, np.nan)
    tx = pd.Series(p["T2M_MAX"], dtype=float).replace(-999, np.nan)
    tm.index = pd.to_datetime(tm.index, format="%Y%m%d")
    tx.index = tm.index
    return tm, tx


def _cooling(cdd):
    return "Low" if cdd < COOLING_MEDIUM_FROM else ("Medium" if cdd < COOLING_HIGH_FROM else "High")


nairobi_yearly_mean_T2M = float(daily_temp.mean())
sites = {"Nairobi (this site)": (daily_temp, t_max),
         "Mombasa (hotter what-if)": _nasa_daily(-4.04, 39.67)}

scenario = []
for name, (tm, tx) in sites.items():
    cdd = float((tm - 18).clip(lower=0).sum())
    daily_peak = area_peak_kw * (1 + TEMP_SENSITIVITY * (tm - nairobi_yearly_mean_T2M))
    print(f"{name}: mean {tm.mean():.1f} C | hottest day {tx.max():.1f} C | "
          f"cooling degree days {cdd:.0f} | cooling {_cooling(cdd)} | "
          f"daily peak mean {daily_peak.mean():.0f} kW, max {daily_peak.max():.0f} kW")
    scenario.append(pd.DataFrame({"site": name, "date": tm.index.strftime("%Y-%m-%d"),
                                  "t2m": tm.values.round(2), "t2m_max": tx.reindex(tm.index).values.round(2),
                                  "daily_peak_kw": daily_peak.values.round(1)}))

weather_scenario = pd.concat(scenario, ignore_index=True)
weather_scenario.to_csv(os.path.join(WORK_DIR, "weather_scenario.csv"), index=False)
print("Saved:", os.path.join(WORK_DIR, "weather_scenario.csv"))

## 8. Productive-use viability

**Adds:** `productive_use` (High / Medium / Low) and `productive_use_reason`.

"Productive use" means using electricity to earn money: milling, welding, a salon, a shop fridge,
tailoring. Mini-grid and utility planners want to know where such customers are, because they use
more and pay more reliably. First simple rules (placeholders):

- **High**: a shop or workshop is already here (map label).
- **Medium**: a large home (over 100 m²) or large unlabelled building within 250 m of a main road.
- **Low**: everything else. Schools, clinics and churches are "anchor" customers (steady users), not businesses.

Scale: High = a business, workshop or agricultural production is already there (map label or manual check);
Medium = a large home or large unlabelled building within PUE_ROAD_M of a main road (space and customers for
a home business); Low = all other buildings. Schools, clinics and churches are 'anchor' customers (steady
users), not businesses. This is a first rule-based screen; the separate OSEAS productive-use challenge maps
these uses in more detail.

In [ ]:
# 8. Productive-use viability: could this building host (or already host) a business that uses electricity?
#    Simple first rules, PLACEHOLDERS to agree with Eliane and the experts.

t = buildings["building_type"]
near_road = buildings["dist_main_road_m"] <= PUE_ROAD_M
rules = [  # (condition, level, reason) - the first rule that matches wins
    (t == "agriculture", "High", "agricultural production (greenhouse/farm)"),
    (t.isin(["commercial", "industrial"]), "High", "business already here (map label)"),
    (t.isin(["education", "health", "religious"]), "Low", "public service: steady 'anchor' customer, not a business"),
    ((t == "unknown_large") & near_road, "Medium", "large unlabelled building near a main road"),
    ((t == "residential") & (buildings["area_m2"] >= PUE_HOME_MIN_M2) & near_road, "Medium",
     "large home near a main road (room for a home business)"),
    (t == "outbuilding", "Low", "small outbuilding"),
]
buildings["productive_use"] = np.select([r[0] for r in rules], [r[1] for r in rules], default="Low")
buildings["productive_use_reason"] = np.select([r[0] for r in rules], [r[2] for r in rules],
                                               default="home or building away from main roads")
print(buildings.groupby(["productive_use", "productive_use_reason"]).size().rename("buildings").to_string())

## 9. Load profiles: hourly and monthly

**Adds:** `daily_kwh` and `load_factor` (average use ÷ peak use: near 1 = steady, near 0 = short spikes),
plus the area's load profiles, saved as `area_load_curve.csv/.png`, `area_monthly.csv/.png` and
`area_hourly_by_month.csv`.

- **Hourly (0-23):** every building's average day, added up for the whole area. The area peak is much
  smaller than the sum of every building's own peak, because buildings do not all peak at the same moment.
  The ratio is the **diversity factor**; planners use it to size the grid.
- **Monthly:** The household load profiles (Nigeria survey, RAMP) have no dates, so on their own they give the same day all year.
  We link months to the NASA temperatures from section 7: a month warmer than average uses a bit more
  (`TEMP_SENSITIVITY`, a placeholder). The yearly total does not change. If the minute-level readings
  (`all_households_minute`) have dates, they could replace this with real monthly variation.

In [ ]:
# 9. Load profiles: per building, the whole area hour by hour (0-24), and month by month.
import matplotlib.pyplot as plt

os.makedirs(WORK_DIR, exist_ok=True)

# ---- a) Per building: how "flat" or "spiky" its use is ----------------------
# load factor = average use / peak use. Near 1 = steady all day (e.g. a fridge). Near 0 = short big spikes.
buildings["daily_kwh"] = buildings["annual_kwh"] / 365
buildings["load_factor"] = (buildings["daily_kwh"] / 24) / buildings["peak_kw"].where(buildings["peak_kw"] > 0)

# ---- b) Average day for EVERY building, hour 0 to 23 (kW in each hour) ------
order = np.argsort(np.asarray(hours) % 24)                                   # make sure hours run 0..23
pattern_shapes = kmeans.cluster_centers_[:, order]
pattern_shapes = pattern_shapes / pattern_shapes.sum(axis=1, keepdims=True)  # each pattern adds up to 1
hourly_by_building = np.zeros((len(buildings), 24))
home = (buildings["profile_source"] == "household_readings").to_numpy()
hourly_by_building[home] = (pattern_shapes[buildings.loc[home, "load_pattern"].astype(int).to_numpy()]
                            * buildings.loc[home, "daily_kwh"].to_numpy()[:, None])
for kind in APPLIANCES:                                                      # RAMP average day x size scale
    is_kind = (buildings["building_type"] == kind).to_numpy()
    hourly_by_building[is_kind] = ramp_hourly_kw[kind][None, :] * buildings.loc[is_kind, "size_scale"].to_numpy()[:, None]

# ---- c) The whole area's average day ----------------------------------------
curve = pd.DataFrame({"homes": hourly_by_building[home].sum(axis=0)}, index=pd.Index(range(24), name="hour"))
for kind in APPLIANCES:
    is_kind = (buildings["building_type"] == kind).to_numpy()
    if is_kind.any():
        curve[kind] = hourly_by_building[is_kind].sum(axis=0)
curve["total"] = curve.sum(axis=1)

area_peak = curve["total"].max()
sum_of_peaks = buildings["peak_kw"].sum()
print(f"Area peak (everyone together, average day): {area_peak:.0f} kW at {int(curve['total'].idxmax())}:00")
print(f"Sum of every building's own peak:           {sum_of_peaks:.0f} kW")
print(f"Diversity factor: {sum_of_peaks / area_peak:.1f}  (buildings do not all peak at the same time,"
      " so the grid needs less than the sum of the peaks)")
print(f"Daily total: {curve['total'].sum():.0f} kWh  |  share used by homes: {curve['homes'].sum() / curve['total'].sum():.0%}")

ax = curve.drop(columns="total").plot.area(figsize=(10, 4), alpha=0.8, linewidth=0)
ax.set_xlabel("Hour of the day"); ax.set_ylabel("kW (whole area)"); ax.set_xticks(range(0, 24, 2))
ax.set_title("Daily load profile of the whole area (average day)")
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, "area_load_curve.png"), dpi=120); plt.show()
curve.round(2).to_csv(os.path.join(WORK_DIR, "area_load_curve.csv"))

# ---- d) Month by month, linked to the weather (section 7) --------------------
# The household profiles have no dates, so on their own they give the same day all year.
# Here hotter months use more (fans, fridges work harder): TEMP_SENSITIVITY per degree above the
# yearly average. PLACEHOLDER. The yearly total stays exactly the same.
days = pd.Series([pd.Period(f"{WEATHER_YEAR}-{m:02d}").days_in_month for m in range(1, 13)],
                 index=pd.Index(range(1, 13), name="month"))
if daily_temp is not None:
    month_temp = daily_temp.groupby(daily_temp.index.month).mean().reindex(days.index)
    month_cdd = (daily_temp - 18).clip(lower=0).groupby(daily_temp.index.month).sum().reindex(days.index)
else:
    month_temp = month_cdd = pd.Series(np.nan, index=days.index)
year_temp = (month_temp * days).sum() / days.sum()
factor = (1 + TEMP_SENSITIVITY * (month_temp - year_temp)).fillna(1.0)
factor = factor * days.sum() / (factor * days).sum()                          # keep the yearly total unchanged

monthly = pd.DataFrame({"days": days, "avg_temp_c": month_temp.round(1), "cooling_degree_days": month_cdd.round(0),
                        "factor": factor.round(3),
                        "kwh": (curve["total"].sum() * days * factor).round(0),
                        "avg_daily_kwh": (curve["total"].sum() * factor).round(0),
                        "peak_kw": (area_peak * factor).round(0)})
monthly.index = [pd.Timestamp(2000, m, 1).strftime("%b") for m in monthly.index]
print()
print(monthly.to_string())
print(f"Year total: {monthly['kwh'].sum() / 1000:.0f} MWh")
monthly.to_csv(os.path.join(WORK_DIR, "area_monthly.csv"))

# Average day in every month (rows = month, columns = hour), for the dashboard and slides
hourly_by_month = pd.DataFrame(np.outer(factor, curve["total"]), index=monthly.index, columns=range(24)).round(1)
hourly_by_month.to_csv(os.path.join(WORK_DIR, "area_hourly_by_month.csv"))

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(monthly.index, monthly["kwh"] / 1000, color="#4c78a8")
ax1.set_ylabel("MWh in the month")
ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly["avg_temp_c"], color="#e45756", marker="o")
ax2.set_ylabel("Average temperature (C)", color="#e45756")
ax1.set_title("Monthly electricity use of the whole area, and temperature")
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, "area_monthly.png"), dpi=120); plt.show()

## 10. Summary, and the answer to Eliane's question

Eliane asked: *were you able to overlay the weather data and the infrastructure data to differentiate
energy consumption across buildings?* The output below shows both.

In [ ]:
# 10. Summary by building type, and the answer to Eliane's question about weather and roads.
summary = buildings.groupby("building_type").agg(
    buildings=("building_id", "size"),
    median_m2=("area_m2", "median"),
    avg_kwh_year=("annual_kwh", "mean"),
    total_mwh_year=("annual_kwh", lambda s: s.sum() / 1000),
    avg_peak_kw=("peak_kw", "mean"),
    cooling=("cooling_category", lambda s: s.mode().iat[0] if s.notna().any() else None),
).round(2)
print(summary.to_string())
summary.to_csv(os.path.join(WORK_DIR, "summary_by_type.csv"))
print()

# Roads: average household electricity by distance to a main road
bands = pd.cut(households["dist_main_road_m"], [0, 250, 500, 1000, np.inf],
               labels=["under 250 m", "250-500 m", "500-1000 m", "over 1000 m"], include_lowest=True)
by_road = households.groupby(bands, observed=False).agg(households=("annual_kwh", "size"),
                                                         avg_kwh_year=("annual_kwh", "mean")).round(0)
print("Households by distance to a main road:")
print(by_road.to_string())
print()
print("Weather: the whole area gets ONE temperature record (the area is small), so weather sets the")
print("cooling level for the area, and shops and clinics go one level higher. Roads are what make")
print("buildings in the same area differ.")

## 10b. What is missing, and how we handle it

When a piece of information is missing, we leave it blank (empty) or mark it as estimated, rather than
inventing a value. This keeps guessed numbers from being mistaken for measured ones. The table below lists
what is missing, how many buildings it affects, and what we do about it.

In [ ]:
# 10b. What is missing, and how we handle it: one row per gap, counts computed from `buildings`.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

n_all = len(buildings)

building_rows = [
    ("No real building-type label",
     int((buildings["type_source"] == "size_rule").sum()),
     "type guessed from size: residential if under LARGE_BUILDING_M2, else unknown_large"),
    ("Type unknown, large building",
     int((buildings["building_type"] == "unknown_large").sum()),
     "kept as unknown_large; generic RAMP profile; to be checked on Google Maps"),
    ("Treated as outbuilding",
     int((buildings["building_type"] == "outbuilding").sum()),
     "0 kWh of its own; counted in the main home"),
    ("No road distance",
     int(buildings["dist_main_road_m"].isna().sum()),
     "road effect set to neutral (factor 1.0)"),
    ("Non-home buildings using placeholder appliance lists",
     int(buildings["profile_source"].astype(str).str.startswith("ramp_").sum()),
     "RAMP placeholder lists, to be replaced with published sources"),
    ("Buildings with no electricity estimate",
     int(buildings["annual_kwh"].isna().sum()),
     "left blank"),
]

rows = [{"item": item, "buildings affected": n,
         "% of all buildings": round(100 * n / n_all, 1),
         "what we do about it": action} for item, n, action in building_rows]

dataset_rows = [
    ("Satellite image provider/date/licence", "unknown, asked Eliane"),
    ("Household profiles: no dates", "monthly pattern estimated from temperature (placeholder sensitivity)"),
    ("Household profiles: source", "RAMP-simulated from a household survey in Nigeria (project dataset)"),
    ("Land use / zoning", "not used yet"),
]
rows += [{"item": item, "buildings affected": "-", "% of all buildings": "-",
          "what we do about it": action} for item, action in dataset_rows]

missing = pd.DataFrame(rows, columns=["item", "buildings affected", "% of all buildings", "what we do about it"])
print(missing.to_string(index=False))

os.makedirs(WORK_DIR, exist_ok=True)
missing.to_csv(os.path.join(WORK_DIR, "missing_data.csv"), index=False)

# Horizontal bar chart of the six building-level rows (% of all buildings)
chart = missing.iloc[:6]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(chart["item"], chart["% of all buildings"].astype(float), color="#4c72b0")
ax.set_xlabel("% of all buildings")
ax.set_title("What is missing (share of all buildings)")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(os.path.join(WORK_DIR, "missing_data.png"), dpi=120)
plt.show()
print("Saved:", os.path.join(WORK_DIR, "missing_data.csv"), "and", os.path.join(WORK_DIR, "missing_data.png"))

## 11. Save and map

Saves the table as a GeoPackage (`.gpkg`, opens in QGIS) and GeoJSON, plus the summary and load-profile
tables and charts from sections 9 and 10, and copies them to your Drive.

| Column | Meaning |
|---|---|
| `building_id` | Our own ID for the building |
| `outline_source` | `model`, `model_new`, `overture_split` or `overture_missed` (step 3c) |
| `new_candidate` | True if only the model found this building: possibly new, needs checking |
| `area_m2` | Roof area in square metres |
| `lon`, `lat` | Centre of the building |
| `building_type` | residential, outbuilding, education, health, religious, commercial, industrial or unknown_large |
| `type_source` | `osm` or `overture` if a map labelled it, `size_rule` if guessed from size |
| `osm_name` | Name of the place, if a map has one |
| `main_building_id` | For outbuildings: the home it belongs to |
| `dist_road_m`, `dist_main_road_m`, `dist_paved_road_m` | Metres to the nearest road / main road / paved road |
| `household_id`, `buildings_in_household` | Which household (compound) a home belongs to, and how many buildings it has |
| `annual_kwh`, `daily_kwh` | Estimated electricity use per year / per day |
| `peak_kw` | Estimated highest hourly demand |
| `load_factor` | Average use ÷ peak (near 1 = steady, near 0 = spiky) |
| `peak_hour` | Hour of the day when use is usually highest |
| `evening_share` | Share of daily use between 18:00 and 22:59 |
| `load_pattern` | Homes: which of the 4 typical daily patterns (0 to 3) |
| `profile_source` | `household_readings`, `part_of_household` (outbuilding) or `ramp_<type>` |
| `road_factor` | Homes: how much the road effect raised (>1) or lowered (<1) use |
| `size_scale` | Non-homes: size compared with the typical building in 6a |
| `cooling_category` | Low, Medium or High cooling need |
| `productive_use`, `productive_use_reason` | High, Medium or Low, and why |

In [ ]:
# 11. Save the deliverable.
import shutil

os.makedirs(WORK_DIR, exist_ok=True)
columns = ["building_id", "outline_source", "new_candidate", "area_m2", "lon", "lat",
           "building_type", "type_source", "osm_name", "main_building_id",
           "dist_road_m", "dist_main_road_m", "dist_paved_road_m",
           "household_id", "buildings_in_household",
           "annual_kwh", "daily_kwh", "peak_kw", "load_factor", "peak_hour", "evening_share",
           "load_pattern", "profile_source", "road_factor", "size_scale",
           "cooling_category", "productive_use", "productive_use_reason", "geometry"]
out = buildings[columns].to_crs(4326)

gpkg_path = os.path.join(WORK_DIR, "buildings_energy.gpkg")
geojson_path = os.path.join(WORK_DIR, "buildings_energy.geojson")
for path in (gpkg_path, geojson_path):
    if os.path.exists(path):
        os.remove(path)
out.to_file(gpkg_path, driver="GPKG", layer="buildings")
out.to_file(geojson_path, driver="GeoJSON")
print("Saved:", gpkg_path, "and", geojson_path)

if DRIVE_OUT_DIR:
    for name in ["buildings_energy.gpkg", "buildings_energy.geojson", "summary_by_type.csv",
                 "area_load_curve.csv", "area_load_curve.png", "area_monthly.csv", "area_monthly.png",
                 "area_hourly_by_month.csv"]:
        shutil.copy2(os.path.join(WORK_DIR, name), DRIVE_OUT_DIR)
    print("Copied the files to:", DRIVE_OUT_DIR)

out.drop(columns="geometry").head(10)

In [ ]:
# 11b. Map: colour = building type. Hover over a building to see its numbers.
show = out.copy()
for col in ["area_m2", "annual_kwh", "peak_kw", "load_factor", "evening_share", "dist_main_road_m"]:
    show[col] = show[col].astype(float).round(2)

building_map = show.explore(
    column="building_type",
    categorical=True,
    tooltip=["building_id", "building_type", "area_m2", "annual_kwh", "peak_kw", "peak_hour",
             "cooling_category", "productive_use", "dist_main_road_m"],
    tiles="Esri.WorldImagery",
    style_kwds={"weight": 1, "fillOpacity": 0.5},
)
building_map.save(os.path.join(WORK_DIR, "buildings_map.html"))
building_map

## 12. Area explorer (dashboard)

**Adds:** `area_explorer.html`, a map you can open in any browser and share.

The area is cut into squares (`CELL_M` = 250 m, change it in Settings). Colour = yearly use (green low,
red high). **Click a square** to see: how many buildings and households, which types, yearly MWh, peak,
an **average day (hour 0-23)** and **month by month**. Hover over a bar to read its value.

Eliane asked for this on the 12 October call: pick any part of the area and see its numbers and load profiles.

In [ ]:
# 12. Area explorer: the area cut into squares. Click a square to see what is inside it.
import folium
from shapely.geometry import box as square

def bar_svg(values, labels, color, unit, width=290, height=90):
    """A tiny bar chart drawn as an SVG picture (works in any browser, no internet needed)."""
    values = np.nan_to_num(np.asarray(values, dtype=float))
    top = values.max() if values.max() > 0 else 1.0
    bw = width / len(values)
    bars = "".join(
        f'<rect x="{i * bw + 1:.1f}" y="{height - 14 - v / top * (height - 24):.1f}" width="{bw - 2:.1f}" '
        f'height="{v / top * (height - 24):.1f}" fill="{color}"><title>{labels[i]}: {v:.1f} {unit}</title></rect>'
        for i, v in enumerate(values))
    ticks = "".join(f'<text x="{i * bw + bw / 2:.1f}" y="{height - 2}" font-size="8" text-anchor="middle">{labels[i]}</text>'
                    for i in range(0, len(values), max(1, len(values) // 6)))
    return (f'<svg width="{width}" height="{height}" style="font-family:sans-serif">{bars}{ticks}'
            f'<text x="0" y="9" font-size="9" fill="#555">max {top:.1f} {unit}</text></svg>')

# ---- Cut the area into squares and add up every building in each square ------
centre = buildings.geometry.centroid
x0, y0 = centre.x.min(), centre.y.min()
col = ((centre.x - x0) // CELL_M).astype(int)
row = ((centre.y - y0) // CELL_M).astype(int)
buildings["cell_id"] = row.astype(str) + "_" + col.astype(str)

cells = []
month_names = list(monthly.index)
for cell_id, idx in buildings.groupby("cell_id").indices.items():
    part = buildings.iloc[idx]
    day = hourly_by_building[idx].sum(axis=0)                                   # kW in each hour, average day
    month_kwh = day.sum() * monthly["days"].to_numpy() * factor.to_numpy()      # kWh in each month
    r, c = map(int, cell_id.split("_"))
    types = part["building_type"].value_counts()
    cells.append({
        "cell_id": cell_id, "buildings": len(part), "households": part["household_id"].nunique(),
        "annual_mwh": part["annual_kwh"].sum() / 1000, "peak_kw": day.max(), "peak_hour": int(day.argmax()),
        "types": ", ".join(f"{k} {v}" for k, v in types.items()),
        "day": day, "months": month_kwh,
        "geometry": square(x0 + c * CELL_M, y0 + r * CELL_M, x0 + (c + 1) * CELL_M, y0 + (r + 1) * CELL_M)})
cells = gpd.GeoDataFrame(cells, crs=crs).to_crs(4326)
print(f"{len(cells)} squares of {CELL_M} m x {CELL_M} m with buildings in them")

# ---- Draw the map ---------------------------------------------------------
lo, hi = cells["annual_mwh"].quantile([0.05, 0.95])
def colour(v):
    t = 0 if hi == lo else min(max((v - lo) / (hi - lo), 0), 1)
    return f"#{int(255 * t):02x}{int(200 * (1 - t)):02x}{int(80 * (1 - t)):02x}"   # green (low) to red (high)

minx, miny, maxx, maxy = cells.total_bounds
dashboard = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=15, tiles=None)
folium.TileLayer("Esri.WorldImagery", name="satellite").add_to(dashboard)
for _, cell in cells.iterrows():
    html = (f"<b>Square {cell.cell_id}</b><br>"
            f"Buildings: {cell.buildings} &nbsp;|&nbsp; households: {cell.households}<br>"
            f"<small>{cell.types}</small><br>"
            f"Yearly use: <b>{cell.annual_mwh:.1f} MWh</b> &nbsp;|&nbsp; peak {cell.peak_kw:.1f} kW at {cell.peak_hour}:00<br>"
            f"<br><b>Average day</b> (kW, hour 0-23)<br>{bar_svg(cell.day, [str(h) for h in range(24)], '#4c78a8', 'kW')}"
            f"<br><b>Month by month</b> (kWh)<br>{bar_svg(cell.months, [m[0] for m in month_names], '#f58518', 'kWh')}")
    folium.GeoJson(cell.geometry.__geo_interface__,
                   style_function=lambda f, c=colour(cell.annual_mwh): {"color": "white", "weight": 0.5,
                                                                         "fillColor": c, "fillOpacity": 0.45},
                   tooltip=f"{cell.buildings} buildings, {cell.annual_mwh:.1f} MWh a year",
                   popup=folium.Popup(html, max_width=320)).add_to(dashboard)
folium.LayerControl().add_to(dashboard)

dashboard_path = os.path.join(WORK_DIR, "area_explorer.html")
dashboard.save(dashboard_path)
if DRIVE_OUT_DIR:
    shutil.copy2(dashboard_path, DRIVE_OUT_DIR)
print("Saved:", dashboard_path, "(open it in a browser and click any square)")
dashboard

## 12b. Example area

A 500 m x 500 m box is placed around the 250 m square that has the most schools, clinics and shops
(education + health + commercial; agriculture ignored). The box is named after the largest labelled school
or clinic inside it. The cell below reports the buildings, labels, totals and load profile of that box, and
also adds a legend, title, school/clinic markers and the example-area box to `area_explorer.html`.

In [ ]:
# 12b. Example area: pick the busiest 250 m square (schools + clinics + shops), build a 500 m box around it.
from shapely.geometry import box as _box

SERVICE_TYPES = ["education", "health", "commercial"]
svc = buildings[buildings["building_type"].isin(SERVICE_TYPES)]
if svc.empty:
    raise ValueError("No education/health/commercial buildings found for the example area.")
best_cell_id = svc.groupby("cell_id").size().sort_values(ascending=False).index[0]

# Centre of that 250 m square, in the metric crs (same origin as section 12).
_centre = buildings.geometry.centroid
x0, y0 = _centre.x.min(), _centre.y.min()
r_c, c_c = map(int, best_cell_id.split("_"))
cx = x0 + (c_c + 0.5) * CELL_M
cy = y0 + (r_c + 0.5) * CELL_M
box_geom = _box(cx - 250, cy - 250, cx + 250, cy + 250)        # 500 m x 500 m

in_box = buildings.geometry.centroid.within(box_geom).to_numpy()
ex = buildings[in_box].copy()
ex_idx = np.where(in_box)[0]

# Name from the largest labelled school or clinic inside the box, if any.
labelled_sc = ex[ex["building_type"].isin(["education", "health"]) & ex["osm_name"].notna()]
area_name = ("Example area — " + labelled_sc.sort_values("area_m2", ascending=False).iloc[0]["osm_name"]
             if len(labelled_sc) else "Example area")
print("Example area:", area_name)

counts = ex["building_type"].value_counts()
print("\nBuildings by type:")
print(counts.to_string())

labelled = ex[ex["osm_name"].notna()][["building_type", "osm_name"]]
print("\nLabelled buildings:")
print(labelled.to_string(index=False) if len(labelled) else "  (none)")

total_mwh = ex["annual_kwh"].sum() / 1000
ex_day = hourly_by_building[ex_idx].sum(axis=0)
ex_peak, ex_peak_hour = float(ex_day.max()), int(ex_day.argmax())
print(f"\nTotal: {total_mwh:.1f} MWh/yr | peak {ex_peak:.1f} kW at {ex_peak_hour}:00")

# Average day stacked by building type, and month-by-month kWh.
day_by_type = {}
for k in list(counts.index):
    m = (buildings["building_type"].to_numpy() == k) & in_box
    day_by_type[k] = hourly_by_building[np.where(m)[0]].sum(axis=0)
ex_hourly = pd.DataFrame(day_by_type, index=pd.Index(range(24), name="hour"))
ex_month = pd.Series((ex_day.sum() * monthly["days"].to_numpy() * factor.to_numpy()).round(0),
                     index=monthly.index, name="kwh")

rows = [("area_name", area_name), ("box_size_m", "500 x 500"), ("buildings", int(len(ex))),
        ("total_mwh_year", round(total_mwh, 2)), ("peak_kw", round(ex_peak, 1)), ("peak_hour", ex_peak_hour)]
rows += [(f"count_{k}", int(v)) for k, v in counts.items()]
pd.DataFrame(rows, columns=["field", "value"]).to_csv(os.path.join(WORK_DIR, "example_area_summary.csv"), index=False)
print("Saved:", os.path.join(WORK_DIR, "example_area_summary.csv"))

# Share with the figures section.
example_area = {"name": area_name, "box_geom_m": box_geom, "ex": ex, "ex_idx": ex_idx, "day": ex_day,
                "hourly_by_type": ex_hourly, "month": ex_month, "counts": counts, "labelled": labelled,
                "peak": ex_peak, "peak_hour": ex_peak_hour, "total_mwh": total_mwh}

# ---- Update area_explorer.html: legend, title, school/clinic markers, example box ----
lo2, hi2 = cells["annual_mwh"].quantile([0.05, 0.95])
def _sq_colour(v):
    tt = 0 if hi2 == lo2 else min(max((v - lo2) / (hi2 - lo2), 0), 1)
    return f"#{int(255 * tt):02x}{int(200 * (1 - tt)):02x}{int(80 * (1 - tt)):02x}"

minx, miny, maxx, maxy = cells.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=15, tiles=None)
folium.TileLayer("Esri.WorldImagery", name="satellite").add_to(m)
for _, cell in cells.iterrows():
    html = (f"<b>Square {cell.cell_id}</b><br>Buildings: {cell.buildings} &nbsp;|&nbsp; "
            f"households: {cell.households}<br><small>{cell.types}</small><br>"
            f"Yearly use: <b>{cell.annual_mwh:.1f} MWh</b> &nbsp;|&nbsp; peak {cell.peak_kw:.1f} kW at {cell.peak_hour}:00<br>"
            f"<br><b>Average day</b> (kW, hour 0-23)<br>{bar_svg(cell.day, [str(h) for h in range(24)], '#4c78a8', 'kW')}"
            f"<br><b>Month by month</b> (kWh)<br>{bar_svg(cell.months, [mn[0] for mn in month_names], '#f58518', 'kWh')}")
    folium.GeoJson(cell.geometry.__geo_interface__,
                   style_function=lambda f, c=_sq_colour(cell.annual_mwh): {"color": "white", "weight": 0.5,
                                                                            "fillColor": c, "fillOpacity": 0.45},
                   tooltip=f"{cell.buildings} buildings, {cell.annual_mwh:.1f} MWh a year",
                   popup=folium.Popup(html, max_width=320)).add_to(m)

# School (blue) and clinic (red) markers with names.
b4326 = buildings.to_crs(4326)
cent4326 = b4326.geometry.centroid
for i in b4326.index:
    bt = b4326.loc[i, "building_type"]
    if bt == "education":
        mcol, lab = "blue", "School"
    elif bt == "health":
        mcol, lab = "red", "Clinic"
    else:
        continue
    nm = b4326.loc[i, "osm_name"] if pd.notna(b4326.loc[i, "osm_name"]) else lab
    folium.CircleMarker([cent4326.loc[i].y, cent4326.loc[i].x], radius=4, color=mcol, fill=True,
                        fill_color=mcol, fill_opacity=0.9, tooltip=f"{lab}: {nm}").add_to(m)

# Example-area box in white.
box4326 = gpd.GeoSeries([box_geom], crs=crs).to_crs(4326).iloc[0]
folium.GeoJson(box4326.__geo_interface__, style_function=lambda f: {"color": "white", "weight": 3, "fill": False},
               tooltip=area_name).add_to(m)

title_html = ('<div style="position:fixed;top:10px;left:60px;z-index:9999;background:white;padding:6px 10px;'
              'border-radius:4px;font-family:sans-serif;font-size:15px;box-shadow:0 1px 4px rgba(0,0,0,.3)">'
              '<b>Area explorer: yearly electricity use by 250 m square</b></div>')
legend_html = ('<div style="position:fixed;bottom:22px;left:22px;z-index:9999;background:white;padding:8px 10px;'
               'border-radius:4px;font-family:sans-serif;font-size:12px;box-shadow:0 1px 4px rgba(0,0,0,.3)">'
               '<b>Yearly electricity use per 250 m square (MWh)</b><br>'
               '<span style="background:linear-gradient(to right,#00c850,#ffc850,#ff0000);display:inline-block;'
               'width:180px;height:12px;vertical-align:middle"></span><br>'
               f'<span style="float:left">{lo2:.1f}</span><span style="float:right">{hi2:.1f}+</span>'
               '<div style="clear:both"></div>'
               '<span style="color:blue">&#9679;</span> school &nbsp; <span style="color:red">&#9679;</span> clinic'
               ' &nbsp; <span style="border:2px solid #888;padding:0 6px"></span> example area</div>')
folium.LayerControl().add_to(m)
m.get_root().html.add_child(folium.Element(title_html))
m.get_root().html.add_child(folium.Element(legend_html))
m.save(os.path.join(WORK_DIR, "area_explorer.html"))
print("Updated:", os.path.join(WORK_DIR, "area_explorer.html"))

## 13. Next steps, placeholders and references

### Decisions from the 12 October call with Eliane
- **Household average (62.4 kWh/month):** fine. Nairobi uses more than the national average.
- **Outbuildings (50 m² within 15 m):** keep for now ("I think that's okay"); check against the informal-housing sizes in the papers Eliane shared.
- **Hourly and monthly load profiles:** added (section 9), and linked to the weather.
- **Placeholders:** replace guesses with numbers from published sources wherever possible (register below).
- **Productive use:** keep it in the table but light. A separate hackathon challenge covers productive use.
- **Dashboard:** added (section 12).
- **To check by eye:** the 886 model-only buildings (`new_candidate = True`) and the 101 large unlabelled buildings.
- **Next:** send June a short update with pictures; prepare the main presentation.

### Placeholder register: every guessed number and what to cite

| Setting | Value now | Status | Source to use or find |
|---|---|---|---|
| `MONTHLY_KWH_MEAN` | 62.4 kWh/month | Agreed with Eliane | Eliane's notebook; compare World Bank MTF Kenya (48.6 kWh/month for grid homes) |
| `OUTBUILDING_MAX_M2`, `ATTACH_DISTANCE_M` | 50 m², 15 m | Agreed with Eliane | Eliane (homes above 50-80 m²); Amos et al. 2021 (120 m² typical Kenyan roof); papers Eliane shared |
| `ROAD_EFFECT` | ±20% | **Guess** | Need a study linking road access to household kWh (Blimpo & Postepska cover connection rates only) |
| `LARGE_BUILDING_M2` | 400 m² | **Guess** | Check by eye on the map; ask the experts |
| 6a appliance lists, `ref_area_m2` | see 6a | **Guess** | Stevanato et al. RAMP profiles for schools and health facilities; published energy-needs surveys of Kenyan clinics and schools |
| `SIZE_EXPONENT`, `SCALE_LIMITS` | 0.8, 0.3x-5x | **Guess** (0.8 mirrors Eliane's `ALPHA`) | Same sources as 6a |
| `COOLING_MEDIUM_FROM`, `COOLING_HIGH_FROM` | 500, 1500 degree days | **Guess** | A published cooling-degree-day classification |
| Cooling base temperature | 18 °C | Common base, confirm | Degree-day convention (18 °C, or 18.3 °C = 65 °F in the US) |
| `TEMP_SENSITIVITY` | +3% per °C | **Guess** | A study of temperature vs household use in East Africa; or a dated household survey |
| `PUE_HOME_MIN_M2`, `PUE_ROAD_M` | 100 m², 250 m | **Guess** (kept light) | The productive-use challenge's definition |
| `POI_MAX_DISTANCE_M`, `MIN_OVERLAP` | 15 m, 30% | Technical choice | Can stay; mention in the methods |

### Still open
1. Where does the 30 cm image come from, and what is its licence? (Esri map tiles are for viewing only.)
2. Do the minute-level readings (`all_households_minute`) have dates? If yes, real monthly profiles.
3. Population per building is not estimated yet.

### Ideas to add later
- Roof material (iron sheet vs. tile) as a wealth signal (Huang et al. 2021).
- Wealth index maps; nightlights.
- Run the same notebook for a second site (e.g. `namabasa.tif`).

### References
- Fobi et al. (2022): predicting household electricity use in Kenya from satellite imagery.
- Huang, Hsiang & Gonzalez-Navarro (2021). Using satellite imagery and deep learning to evaluate the impact of anti-poverty programs (roof materials in Siaya, Kenya).
- Blimpo & Postepska: electricity connection rates are much higher in places with paved roads (47% vs 20%).
- Stevanato et al.: RAMP load profiles for sub-Saharan Africa, including schools and health facilities (Zenodo, CC-BY).
- World Bank ESMAP Multi-Tier Framework, Kenya: grid-connected homes average 48.6 kWh a month.
- IEA: minimum electricity access is 250 kWh a year (rural) and 500 kWh a year (urban) per household.
- Amos et al. (2021). Improving household agriculture with roof-harvested rainwater: a case study in Sydney and Nairobi. *Water* 13(20), 2920 (uses 120 m² as a typical Kenyan single-family roof). https://www.mdpi.com/2073-4441/13/20/2920
- Paper Eliane shared: https://www.sciencedirect.com/science/article/pii/S2352728521000130
- Lombardi et al. (2019), RAMP: generating high-resolution multi-energy load profiles for remote areas. https://github.com/RAMP-project/RAMP
- NASA POWER daily weather data: https://power.larc.nasa.gov
- OpenStreetMap (ODbL) and Overture Maps (buildings and places, ODbL / CDLA). Derived data must keep these licences.

In [ ]:
# Data inventory: how many records of each dataset were actually used (from variables already in memory).
print("DATA INVENTORY")
print("-" * 48)

try:
    vc = buildings["outline_source"].value_counts()
    print(f"Building outlines used (total {len(buildings)}):")
    for src, n in vc.items():
        print(f"    {src}: {n}")
except Exception:
    print("Building outlines used: not available")

try:
    print(f"OpenStreetMap labelled places: {len(osm)}")
except Exception:
    print("OpenStreetMap labelled places: not available")

try:
    print(f"Overture places used: {len(ovt_places)}")
except Exception:
    print("Overture places used: not available")

try:
    print(f"Labels matched to a building: {int(buildings['type_source'].isin(['osm', 'overture']).sum())}")
except Exception:
    print("Labels matched to a building: not available")

try:
    print(f"Road pieces: {len(roads)}  (main roads: {int(roads['is_main'].sum())})")
except Exception:
    print("Road pieces: not available")

try:
    print(f"Survey-based household profiles used (Nigeria, RAMP): {len(X_norm)}")
except Exception:
    print("Survey-based household profiles used (Nigeria, RAMP): not available")

try:
    print(f"NASA POWER days of weather: {len(daily_temp)}")
except Exception:
    print("NASA POWER days of weather: not available")

# Figures for the presentation

Slide-ready PNGs (1600x900, 16:9) are written to `building_table_out/figures/`. Each building type keeps the
same colour across every figure, titles are in plain words, axes are labelled with units, and a short source
line sits at the bottom-left.

In [ ]:
# =====================================================================================
# FIGURES FOR THE MIDPOINT SLIDES — version 2 (each picture sized to its slide spot)
# Paste ALL of this into ONE new cell at the very BOTTOM of building_table_v5.ipynb
# (replace the cell you added last time), after "Run All" has finished. Then run it.
# It only reads the results and draws pictures; it changes nothing else.
# =====================================================================================
import os, textwrap, warnings
warnings.filterwarnings("ignore", message="The GeoSeries you are attempting to plot is empty")
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt, requests
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.transforms import blended_transform_factory
from shapely.geometry import box as _box, Point
from shapely.ops import nearest_points
try:
    import contextily as cx
except ImportError:
    cx = None
    print("contextily is not installed: maps will have no satellite photo. Fix: run  %pip install contextily  in a new cell, then re-run this cell.")

# ---------------------------------------------------------------- PART 1: SETUP
FIG_DIR = os.path.join(WORK_DIR, "figures_v3"); os.makedirs(FIG_DIR, exist_ok=True)
DPI = 200                                   # sharp on a projector; sizes below are in inches = slide size / 100
plt.rcParams.update({"font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "xtick.labelsize": 12,
                     "ytick.labelsize": 12, "legend.fontsize": 12, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
SITE_BBOX = (36.861042, -1.190780, 36.899746, -1.160067)        # the study box (W, S, E, N), degrees
NAMES = {"residential": "Homes", "outbuilding": "Outbuildings", "unknown_large": "Large, use unknown",
         "commercial": "Shops / businesses", "industrial": "Workshops", "education": "Schools",
         "religious": "Churches / mosques", "agriculture": "Greenhouses / farms", "health": "Clinics"}
COLORS = {"residential": "#4A76A8", "outbuilding": "#A9C4E2", "unknown_large": "#B8ADA6", "commercial": "#F28E2B",
          "industrial": "#76B7B2", "education": "#59A14F", "religious": "#B07AA1", "agriculture": "#EDC948",
          "health": "#E15759"}
ORDER = ["residential", "outbuilding", "unknown_large", "commercial", "industrial", "education", "religious",
         "agriculture", "health"]
GROUPS = [("Homes", ["residential"], "#4A76A8"),
          ("Shops, schools, clinics, churches", ["commercial", "industrial", "education", "health", "religious"], "#F28E2B"),
          ("Large unknown + greenhouses", ["unknown_large", "agriculture"], "#B8ADA6")]
HALO = [pe.withStroke(linewidth=3, foreground="white")]
SRC = "Source: Overture Maps 2026-09-23.1, OpenStreetMap, NASA POWER; model results"

def plain(t): return NAMES.get(t, str(t))
def col(t): return COLORS.get(t, "#999999")

def finish(fig, name, source=SRC):
    fig.text(0.006, 0.008, source, fontsize=7.5, color="#777777")
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)
    print("Saved:", path)

def add_satellite(ax, alpha=1.0, zoom="auto"):
    if cx is None:
        return
    for z in ([zoom, 17] if zoom != "auto" else ["auto"]):
        try:
            cx.add_basemap(ax, crs=buildings.crs, source=cx.providers.Esri.WorldImagery, attribution=False,
                           alpha=alpha, zoom=z, zorder=0)
            return
        except Exception as err:
            last = err
    print("  (no satellite photo:", str(last)[:80], ")")

def shade_day(ax, labels=True):
    ax.axvspan(8, 17, color="#EEEEEE", zorder=0); ax.axvspan(18, 22, color="#FFF2C6", zorder=0)
    if labels:
        tr = ax.get_xaxis_transform()
        ax.text(12.5, 1.01, "daytime", transform=tr, ha="center", va="bottom", color="#777777", fontsize=11)
        ax.text(20, 1.01, "evening", transform=tr, ha="center", va="bottom", color="#A07800", fontsize=11)

def scale_bar(ax, metres, label):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x, y = x0 + 0.04 * (x1 - x0), y0 + 0.05 * (y1 - y0)
    ax.plot([x, x + metres], [y, y], color="white", lw=7, solid_capstyle="butt", zorder=9)
    ax.plot([x, x + metres], [y, y], color="black", lw=3.5, solid_capstyle="butt", zorder=10)
    ax.text(x + metres / 2, y + 0.02 * (y1 - y0), label, ha="center", va="bottom", fontsize=12,
            path_effects=HALO, zorder=10)

def group_day(mask):
    """Average day (kW per hour) of the buildings in `mask`, added up into the 3 GROUPS."""
    return {g: hourly_by_building[mask & np.isin(btype, kinds)].sum(axis=0) for g, kinds, _ in GROUPS}

def stack_day(ax, days, peak_label=True):
    shade_day(ax)
    names = [g for g, _, _ in GROUPS if days[g].sum() > 0]
    ax.stackplot(range(24), [days[g] for g in names], colors=[c for g, _, c in GROUPS if g in names],
                 labels=names, alpha=0.95, zorder=2)
    total = sum(days[g] for g in names)
    ax.set_xlim(0, 23); ax.set_ylim(0, total.max() * 1.18); ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel("Hour of the day"); ax.set_ylabel("kW")
    ax.legend(loc="upper left", frameon=True, framealpha=0.95, fontsize=11)
    return total

assert len(hourly_by_building) == len(buildings), "Run the whole notebook again first (Run All)."
btype = buildings["building_type"].astype(str).to_numpy()
types_here = [t for t in ORDER if t in set(btype)] + sorted(set(btype) - set(ORDER))
centres = buildings.geometry.centroid
site_total = hourly_by_building.sum(axis=0)
AREA_PEAK, AREA_PEAK_HOUR = site_total.max(), int(site_total.argmax())
site_poly = gpd.GeoSeries([_box(*SITE_BBOX)], crs=4326).to_crs(buildings.crs).iloc[0]
SITE_KM2 = site_poly.area / 1e6
sx0, sy0, sx1, sy1 = site_poly.bounds
N_HH = int(buildings["household_id"].nunique())
NAME_COL = next((c for c in ["osm_name", "name", "place_name", "poi_name"] if c in buildings.columns), None)
counts = pd.Series(btype).value_counts()

# The example 500 m block (slide 8). Leave None to pick it automatically near "Jacaranda".
EXAMPLE_CENTER_LONLAT = None        # or e.g. (36.8930, -1.1850)
EX = 500
def _score(x, y):
    inside = centres.within(_box(x - EX / 2, y - EX / 2, x + EX / 2, y + EX / 2)).to_numpy()
    kinds = set(btype[inside]) & {"commercial", "education", "health", "religious", "industrial", "agriculture"}
    return len(kinds) * 1000 + np.isin(btype[inside], list(kinds)).sum() * 10 + inside.sum() / 100
if EXAMPLE_CENTER_LONLAT:
    p = gpd.GeoSeries([Point(*EXAMPLE_CENTER_LONLAT)], crs=4326).to_crs(buildings.crs).iloc[0]; EX_X, EX_Y = p.x, p.y
else:
    hit = buildings[buildings[NAME_COL].astype(str).str.contains("Jacaranda", case=False, na=False)] if NAME_COL else []
    if len(hit):
        a = hit.geometry.centroid.iloc[0]
        cands = [(a.x + dx, a.y + dy) for dx in range(-200, 201, 50) for dy in range(-200, 201, 50)]
    else:
        cands = [(x, y) for x in np.arange(sx0 + 250, sx1 - 250, 125) for y in np.arange(sy0 + 250, sy1 - 250, 125)]
    EX_X, EX_Y = max(cands, key=lambda c: _score(*c))
EX_BOX = _box(EX_X - EX / 2, EX_Y - EX / 2, EX_X + EX / 2, EX_Y + EX / 2)
in_block = centres.within(EX_BOX).to_numpy()
print(f"Site {SITE_KM2:.1f} km² | {len(buildings):,} buildings | {N_HH:,} households | peak {AREA_PEAK:,.0f} kW at {AREA_PEAK_HOUR}:00 "
      f"| {buildings['annual_kwh'].sum() / 1000:,.0f} MWh/yr")

# ---------------------------------------------------------------- PART 2: fig_v3_01_site_map (slides 1-2)
try:
    fig = plt.figure(figsize=(16, 9))
    ax = fig.add_axes([0.005, 0.03, 0.70, 0.95])
    for t in types_here:
        buildings[btype == t].plot(ax=ax, color=col(t), edgecolor="none", zorder=3)
    ax.set_xlim(sx0, sx1); ax.set_ylim(sy0, sy1); ax.set_anchor("W")
    add_satellite(ax)
    ax.plot(*EX_BOX.exterior.xy, color="white", lw=3, ls="--", zorder=5)
    ax.text(EX_BOX.bounds[0] + 15, EX_BOX.bounds[3] - 15, "Example\nblock", color="white", fontsize=13, va="top",
            fontweight="bold", path_effects=[pe.withStroke(linewidth=3, foreground="black")], zorder=6)
    scale_bar(ax, 500, "500 m"); ax.set_axis_off()
    key = fig.add_axes([0.72, 0.08, 0.27, 0.86]); key.set_axis_off()
    key.text(0, 1, "Kahawa West,\nNairobi–Kiambu border", fontsize=20, fontweight="bold", va="top")
    key.text(0, 0.86, f"{SITE_KM2:.1f} km² · {len(buildings):,} buildings", fontsize=15, va="top", color="#444444")
    for i, t in enumerate(types_here):
        y = 0.74 - i * 0.065
        key.add_patch(plt.Rectangle((0, y - 0.022), 0.07, 0.04, color=col(t), transform=key.transAxes))
        key.text(0.1, y, f"{plain(t)}  ({counts[t]:,})", fontsize=14, va="center")
    finish(fig, "fig_v3_01_site_map.png", "Source: Overture Maps 2026-09-23.1, OpenStreetMap; imagery Esri World Imagery; model results")
except Exception as err:
    print("fig_v3_01 failed:", err)

# ---------------------------------------------------------------- PART 3: fig_v3_02_counts (slide 6, left; 850 x 480)
try:
    st = (pd.DataFrame({"t": btype, "a": buildings["area_m2"].to_numpy()}).groupby("t")["a"]
          .agg(n="size", med="median").sort_values("n"))
    fig = plt.figure(figsize=(8.5, 4.8))
    ax = fig.add_axes([0.25, 0.1, 0.5, 0.76])
    y = np.arange(len(st))
    ax.barh(y, st["n"], color=[col(t) for t in st.index], height=0.7)
    for yi, n_ in zip(y, st["n"]):
        ax.text(n_ + st["n"].max() * 0.02, yi, f"{int(n_):,}", va="center", fontsize=12)
    ax.set_yticks(y); ax.set_yticklabels([plain(t) for t in st.index], fontsize=12.5)
    ax.set_xlim(0, st["n"].max() * 1.18); ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    tr = blended_transform_factory(ax.transAxes, ax.transData)
    ax.text(1.08, len(st) - 0.3, "typical roof", transform=tr, fontsize=11.5, color="#555555", fontweight="bold")
    for yi, m in zip(y, st["med"]):
        ax.text(1.08, yi, f"{m:,.0f} m²", transform=tr, va="center", fontsize=12, color="#333333")
    ax.set_title(f"Buildings by type  ({N_HH:,} households)", fontsize=14, loc="left", x=-0.45)
    finish(fig, "fig_v3_02_counts.png", "Source: Overture Maps, OpenStreetMap; model results. Typical roof = median.")
except Exception as err:
    print("fig_v3_02 failed:", err)

# ---------------------------------------------------------------- PART 4: fig_v3_03_area_day (slide 6, right; 850 x 480)
try:
    fig = plt.figure(figsize=(8.5, 4.8))
    ax = fig.add_axes([0.1, 0.17, 0.87, 0.7])
    tot = stack_day(ax, group_day(np.ones(len(buildings), bool)))
    mh = int(np.argmax(tot[4:11]) + 4)
    ax.annotate(f"Evening peak {tot.max():,.0f} kW", (AREA_PEAK_HOUR, tot.max()), (10.3, tot.max() * 1.0),
                fontsize=12, arrowprops=dict(arrowstyle="->"), va="center")
    ax.annotate(f"Morning {tot[mh]:,.0f} kW", (mh, tot[mh]), (9.3, tot[mh] * 0.72), fontsize=12,
                arrowprops=dict(arrowstyle="->"), va="center")
    ax.set_ylabel("kW, whole site")
    ax.set_title("Whole site, average day", fontsize=14, loc="left", pad=16)
    finish(fig, "fig_v3_03_area_day.png")
except Exception as err:
    print("fig_v3_03 failed:", err)

# ---------------------------------------------------------------- PART 5: fig_v3_04_grid (dashboard view; 900 x 720)
try:
    if "cells" in globals() and "annual_mwh" in cells.columns:
        grid = cells.to_crs(buildings.crs)
    else:
        g = pd.DataFrame({"x": centres.x // CELL_M, "y": centres.y // CELL_M, "k": buildings["annual_kwh"].to_numpy()})
        a = g.groupby(["x", "y"])["k"].sum().reset_index()
        grid = gpd.GeoDataFrame({"annual_mwh": a["k"] / 1000}, crs=buildings.crs, geometry=[
            _box(x * CELL_M, y * CELL_M, (x + 1) * CELL_M, (y + 1) * CELL_M) for x, y in zip(a["x"], a["y"])])
    fig = plt.figure(figsize=(9, 7.2))
    ax = fig.add_axes([0.01, 0.22, 0.98, 0.76])
    vmax = grid["annual_mwh"].quantile(0.98)
    grid.plot(ax=ax, column="annual_mwh", cmap="YlOrRd", vmin=0, vmax=vmax, alpha=0.82, edgecolor="white",
              linewidth=0.5, zorder=2)
    ax.set_xlim(sx0, sx1); ax.set_ylim(sy0, sy1)
    add_satellite(ax, alpha=0.6)
    ax.plot(*EX_BOX.exterior.xy, color="black", lw=3.5, zorder=5)
    ax.text(EX_X, EX_Y, "Example\nblock", ha="center", va="center", fontsize=12, fontweight="bold",
            path_effects=HALO, zorder=6)
    scale_bar(ax, 500, "500 m"); ax.set_axis_off()
    cax = fig.add_axes([0.2, 0.13, 0.6, 0.026])
    cb = fig.colorbar(plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(0, vmax)), cax=cax, orientation="horizontal")
    cb.set_label(f"Electricity used per year in each {CELL_M} m × {CELL_M} m square (MWh)", fontsize=12)
    finish(fig, "fig_v3_04_grid.png")
except Exception as err:
    print("fig_v3_04 failed:", err)

# ---------------------------------------------------------------- PART 6: fig_v3_05a_block_map (slide 8, left; 1100 x 760)
try:
    blk, bt = buildings[in_block], btype[in_block]
    nb = pd.Series(bt).value_counts()
    fig = plt.figure(figsize=(11, 7.6))
    ax = fig.add_axes([0.005, 0.03, 0.66, 0.95])
    for t in types_here:
        blk[bt == t].plot(ax=ax, color=col(t), edgecolor="white", linewidth=0.3, zorder=3)
    ax.set_xlim(*EX_BOX.bounds[0::2]); ax.set_ylim(*EX_BOX.bounds[1::2]); ax.set_anchor("W")
    add_satellite(ax, zoom=18)
    scale_bar(ax, 100, "100 m"); ax.set_axis_off()
    places = []
    if NAME_COL:
        nm = blk[blk[NAME_COL].notna() & (blk[NAME_COL].astype(str).str.strip() != "")
                 & ~blk["building_type"].isin(["residential", "outbuilding"])].copy()
        prio = {"health": 0, "education": 1, "religious": 2, "industrial": 3, "commercial": 4, "agriculture": 5, "unknown_large": 6}
        nm["p"] = nm["building_type"].map(prio).fillna(9)
        nm = nm.sort_values(["p", "area_m2"], ascending=[True, False]).drop_duplicates(NAME_COL).head(6)
        for i, (_, r) in enumerate(nm.iterrows(), 1):
            c = r.geometry.centroid
            ax.scatter(c.x, c.y, s=330, color="white", edgecolor="black", linewidth=1.5, zorder=7)
            ax.text(c.x, c.y, str(i), ha="center", va="center", fontsize=11, fontweight="bold", zorder=8)
            places.append((i, str(r[NAME_COL]), plain(r["building_type"])))
    key = fig.add_axes([0.68, 0.03, 0.31, 0.93]); key.set_axis_off()
    key.text(0, 1, "Building types", fontsize=14, fontweight="bold", va="top")
    y = 0.93
    for t in types_here:
        if nb.get(t, 0):
            key.add_patch(plt.Rectangle((0, y - 0.016), 0.06, 0.03, color=col(t), transform=key.transAxes))
            key.text(0.09, y, f"{plain(t)} ({nb[t]})", fontsize=12, va="center"); y -= 0.045
    if places:
        y -= 0.03; key.text(0, y, "Numbered places", fontsize=14, fontweight="bold", va="top"); y -= 0.06
        for i, name, kind in places:
            lines = textwrap.wrap(f"{name} ({kind.lower()})", 34)[:2]
            if y - 0.033 * len(lines) < 0.02:
                break
            key.text(0, y, f"{i}", fontsize=12, fontweight="bold", va="top")
            key.text(0.08, y, "\n".join(lines), fontsize=11, va="top"); y -= 0.033 * len(lines) + 0.018
    finish(fig, "fig_v3_05a_block_map.png", "Source: Overture Maps, OpenStreetMap, Overture Places; imagery Esri World Imagery")
except Exception as err:
    print("fig_v3_05a failed:", err)

# ---------------------------------------------------------------- PART 7: fig_v3_05b_block_day (slide 8, right; 620 x 360)
try:
    fig = plt.figure(figsize=(6.2, 3.6))
    ax = fig.add_axes([0.13, 0.17, 0.84, 0.68])
    tot = stack_day(ax, group_day(in_block))
    pk, ph = tot.max(), int(tot.argmax())
    ax.annotate(f"{pk:.0f} kW at {ph}:00", (ph, pk), (9, pk * 1.02), fontsize=11.5, arrowprops=dict(arrowstyle="->"), va="center")
    ax.legend().remove()
    ax.set_title("This block's average day", fontsize=13, loc="left", pad=14)
    finish(fig, "fig_v3_05b_block_day.png", "Model results")
    hh = buildings.loc[in_block & (btype == "residential"), "household_id"].nunique()
    print(f"BLOCK: { {k: int(v) for k, v in pd.Series(btype[in_block]).value_counts().items()} } | households {hh} | "
          f"{buildings.loc[in_block, 'annual_kwh'].sum() / 1000:.0f} MWh/yr | peak {pk:.0f} kW at {ph}:00 | "
          f"≈{int(np.ceil(pk / 0.8 / 5) * 5)} kVA")
except Exception as err:
    print("fig_v3_05b failed:", err)

# ---------------------------------------------------------------- PART 8: fig_v3_06_share_then_avg (slide 7, left; 1040 x 585)
try:
    nbld = buildings[buildings["building_type"] != "outbuilding"]
    g = nbld.groupby("building_type")["annual_kwh"].agg(total="sum", mean="mean", n="size")
    g["share"] = 100 * g["total"] / g["total"].sum()
    g = g.sort_values("share")
    fig = plt.figure(figsize=(10.4, 5.85))
    a1 = fig.add_axes([0.2, 0.12, 0.34, 0.7]); a2 = fig.add_axes([0.6, 0.12, 0.34, 0.7])
    y = np.arange(len(g))
    a1.barh(y, g["share"], color=[col(t) for t in g.index], height=0.7)
    for yi, v in zip(y, g["share"]):
        a1.text(v + 2, yi, f"{v:.1f}%", va="center", fontsize=12)
    a1.set_yticks(y); a1.set_yticklabels([plain(t) for t in g.index], fontsize=12.5)
    a1.set_xlim(0, 115); a1.set_xlabel("% of the site's yearly kWh")
    a1.set_title(f"1. Share of total use\n({g['total'].sum() / 1000:,.0f} MWh a year)", fontsize=13.5, loc="left")
    bars = a2.barh(y, g["mean"], color=[col(t) for t in g.index], height=0.7)
    for yi, (t, v) in zip(y, g["mean"].items()):
        a2.text(v + g["mean"].max() * 0.02, yi, f"{v:,.0f}" + ("*" if t == "agriculture" else ""), va="center", fontsize=12)
        if t == "agriculture":
            bars[list(g.index).index(t)].set_hatch("//"); bars[list(g.index).index(t)].set_edgecolor("#7A6A20")
    a2.set_yticks(y); a2.set_yticklabels([])
    a2.set_xlim(0, g["mean"].max() * 1.25); a2.set_xlabel("kWh per building per year")
    a2.set_title("2. Average individual building\nannual consumption", fontsize=13.5, loc="left")
    fig.suptitle(f"Homes use {g['share'].get('residential', 0):.0f}% of the electricity: each uses little, "
                 f"but there are {int(g['n'].get('residential', 0)):,} of them", fontsize=14.5, x=0.02, ha="left", y=0.97)
    finish(fig, "fig_v3_06_share_then_avg.png", "* placeholder appliance list. Homes: survey profiles (Nigeria); others: RAMP "
           "appliance lists (placeholders). Outbuildings are counted with their home.")
except Exception as err:
    print("fig_v3_06 failed:", err)

# ---------------------------------------------------------------- PART 9: fig_v3_07_type_shapes (slide 7, right; 656 x 585)
try:
    fig = plt.figure(figsize=(6.56, 5.85))
    ax = fig.add_axes([0.13, 0.25, 0.83, 0.6])
    shade_day(ax)
    shown = []
    for t in ["residential", "commercial", "education", "health"]:
        d = hourly_by_building[btype == t].sum(axis=0)
        if d.sum() > 0:
            ax.plot(range(24), 100 * d / d.sum(), color=col(t), lw=3, label=plain(t), zorder=3); shown.append(t)
    ax.set_xlim(0, 23); ax.set_xticks(range(0, 24, 3)); ax.set_xlabel("Hour of the day")
    ax.set_ylabel("% of that type's daily use")
    ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.17), ncol=2, frameon=False, fontsize=12)
    ax.set_title("Homes peak in the evening;\nshops, schools, clinics in the daytime", fontsize=13.5, loc="left", pad=18)
    finish(fig, "fig_v3_07_type_shapes.png", "Homes: survey profiles (Nigeria); others: RAMP (placeholders)")
except Exception as err:
    print("fig_v3_07 failed:", err)

# ---------------------------------------------------------------- PART 10: fig_v3_08_weather (slide 9; 1060 x 600)
try:
    def nasa_daily(lat, lon):
        prm = {"parameters": "T2M,T2M_MAX", "community": "RE", "longitude": lon, "latitude": lat,
               "start": f"{WEATHER_YEAR}0101", "end": f"{WEATHER_YEAR}1231", "format": "JSON"}
        r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", params=prm, timeout=120)
        r.raise_for_status(); p = r.json()["properties"]["parameter"]
        mean = pd.Series(p["T2M"], dtype=float).replace(-999, np.nan); mx = pd.Series(p["T2M_MAX"], dtype=float).replace(-999, np.nan)
        mean.index = mx.index = pd.to_datetime(mean.index, format="%Y%m%d")
        return mean, mx
    c4326 = gpd.GeoSeries([site_poly.centroid], crs=buildings.crs).to_crs(4326).iloc[0]
    s_mean, s_max = nasa_daily(round(c4326.y, 4), round(c4326.x, 4))
    m_mean, m_max = nasa_daily(-4.0435, 39.6682)                                  # Mombasa
    s_cdd, m_cdd = (s_mean - 18).clip(lower=0).sum(), (m_mean - 18).clip(lower=0).sum()
    base = s_mean.mean()
    s_pk = AREA_PEAK * (1 + TEMP_SENSITIVITY * (s_mean - base)); m_pk = AREA_PEAK * (1 + TEMP_SENSITIVITY * (m_mean - base))
    fig = plt.figure(figsize=(10.6, 6.0))
    a1 = fig.add_axes([0.07, 0.3, 0.4, 0.58]); a2 = fig.add_axes([0.58, 0.3, 0.4, 0.58])
    for ser, c in [(m_mean, "#D62728"), (s_mean, "#1F5FA8")]:
        a1.plot(ser.index, ser, color=c, lw=1.3); a1.fill_between(ser.index, 18, ser, where=ser > 18, color=c, alpha=0.25)
    a1.axhline(18, color="black", lw=1, ls="--")
    a1.text(s_mean.index[-1], 18.2, "18 °C", ha="right", va="bottom", fontsize=11, path_effects=HALO)
    a1.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 3, 5, 7, 9, 11])); a1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    a1.set_ylabel("Daily average temperature (°C)")
    a1.set_title(f"MEASURED ({WEATHER_YEAR}): days above 18 °C\nshaded area = cooling score", fontsize=13, loc="left")
    a1.legend(handles=[Line2D([], [], color="#1F5FA8", lw=3, label=f"This site: {s_cdd:,.0f} degree-days (≈{s_cdd / 365:.1f} °C a day)"),
                       Line2D([], [], color="#D62728", lw=3, label=f"Mombasa: {m_cdd:,.0f} degree-days (≈{m_cdd / 365:.1f} °C a day)")],
              loc="upper left", bbox_to_anchor=(-0.05, -0.1), frameon=False, fontsize=11.5)
    a2.scatter(s_max, s_pk, s=14, color="#1F5FA8", alpha=0.7); a2.scatter(m_max, m_pk, s=14, color="#D62728", alpha=0.7)
    a2.set_xlabel("Daily maximum temperature (°C)"); a2.set_ylabel("Modelled daily peak, whole site (kW)")
    a2.set_title("WHAT-IF: same buildings, hotter weather", fontsize=13, loc="left")
    a2.legend(handles=[Line2D([], [], marker="o", ls="", color="#1F5FA8", label=f"This site's weather (avg {s_pk.mean():,.0f} kW)"),
                       Line2D([], [], marker="o", ls="", color="#D62728", label=f"Mombasa's weather (avg {m_pk.mean():,.0f} kW)")],
              loc="upper left", bbox_to_anchor=(-0.05, -0.17), frameon=False, fontsize=11.5)
    a2.text(0.98, 0.03, f"Slope = our assumed +{TEMP_SENSITIVITY:.0%} demand per °C (placeholder)", fontsize=10, color="#555555",
            transform=a2.transAxes, ha="right")
    finish(fig, "fig_v3_08_weather.png", f"Source: NASA POWER daily temperatures, {WEATHER_YEAR}; peak = model results")
    print(f"WEATHER: site {s_cdd:.0f} CDD (mean {s_mean.mean():.1f} °C) | Mombasa {m_cdd:.0f} CDD (mean {m_mean.mean():.1f} °C) | "
          f"avg daily peak {s_pk.mean():.0f} -> {m_pk.mean():.0f} kW (+{m_pk.mean() / s_pk.mean() - 1:.0%})")
except Exception as err:
    print("fig_v3_08 failed:", err)

# ---------------------------------------------------------------- PART 11: fig_v3_10_rules_closeup (slide 10; 1728 x 560)
try:
    has_roads = "roads" in globals() and roads is not None and "is_main" in roads.columns
    W, H = 200, 88                                                           # window in metres (fits the slide strip)
    out_mask = buildings["building_type"] == "outbuilding"
    n_out = buildings.loc[out_mask, "main_building_id"].value_counts()
    if has_roads:
        mr = roads[roads["is_main"]].to_crs(buildings.crs)
        road_union = mr.geometry.union_all() if hasattr(mr.geometry, "union_all") else mr.geometry.unary_union
    cand = buildings[(buildings["building_type"] == "residential") & (buildings["productive_use"] == "Medium")
                     & buildings["dist_main_road_m"].between(30, 70)]
    if not len(cand):
        cand = buildings[(buildings["building_type"] == "residential") & buildings["dist_main_road_m"].between(30, 70)]
    best, best_s = None, -1
    for _, r in cand.head(300).iterrows():
        c = r.geometry.centroid
        rp = nearest_points(c, road_union)[1] if has_roads else c
        cxm, cym = (c.x + rp.x) / 2, (c.y + rp.y) / 2
        w = _box(cxm - W / 2, cym - H / 2, cxm + W / 2, cym + H / 2)
        ins = centres.within(w).to_numpy()
        s = 3 * buildings.loc[ins, "building_id"].isin(n_out[n_out >= 1].index).sum() + ins.sum() / 10
        if s > best_s:
            best, best_s, target, win = (cxm, cym), s, r, w
    fig = plt.figure(figsize=(17.28, 5.6))
    ax = fig.add_axes([0.003, 0.03, 0.66, 0.95])
    view = buildings[buildings.intersects(win)]
    vt = view["building_type"].astype(str)
    for t in types_here:
        view[vt == t].plot(ax=ax, color=col(t), edgecolor="white", linewidth=0.6, zorder=3)
    if has_roads:
        mr[mr.intersects(win.buffer(100))].plot(ax=ax, color="#2B2B2B", linewidth=9, zorder=2)
    geo = buildings.set_index("building_id").geometry
    for _, r in view[vt == "outbuilding"].iterrows():
        if pd.notna(r["main_building_id"]) and r["main_building_id"] in geo.index:
            a, b = r.geometry.centroid, geo[r["main_building_id"]].centroid
            ax.plot([a.x, b.x], [a.y, b.y], color="black", lw=2, zorder=4)
    for hid, grp in view[vt == "residential"].groupby("household_id"):
        if len(grp) >= 2:
            hull = grp.geometry.buffer(2.5).union_all() if hasattr(grp.geometry, "union_all") else grp.geometry.buffer(2.5).unary_union
            hull = hull.convex_hull
            ax.plot(*hull.exterior.xy, color="white", lw=2.2, ls="--", zorder=5)
    c = target.geometry.centroid
    if has_roads:
        rp = nearest_points(c, road_union)[1]
        ax.annotate("", xy=(rp.x, rp.y), xytext=(c.x, c.y), zorder=7,
                    arrowprops=dict(arrowstyle="-|>", lw=3.5, color="#FFD400", path_effects=[pe.withStroke(linewidth=6, foreground="black")]))
        ax.text((c.x + rp.x) / 2 + 4, (c.y + rp.y) / 2, f"{target['dist_main_road_m']:.0f} m to\nmain road", fontsize=13,
                fontweight="bold", path_effects=HALO, zorder=8, va="center")
    lbl = ("Large home near main road:\nproductive use Medium" if target["productive_use"] == "Medium"
           else f"Home, productive use {target['productive_use']}")
    ax.annotate(lbl, (c.x, c.y), (c.x - 0.33 * W, c.y - 0.3 * H), fontsize=12.5,
                path_effects=HALO, zorder=8, arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.set_xlim(*win.bounds[0::2]); ax.set_ylim(*win.bounds[1::2]); ax.set_anchor("W")
    add_satellite(ax, zoom=18); ax.set_axis_off()
    key = fig.add_axes([0.68, 0.06, 0.31, 0.9]); key.set_axis_off()
    key.text(0, 1, "How to read it", fontsize=15, fontweight="bold", va="top")
    items = [(Patch(color=col(t)), plain(t)) for t in types_here if (vt == t).any()] + [
        (Line2D([], [], color="black", lw=2), "outbuilding → the home it belongs to"),
        (Line2D([], [], color="#888888", lw=2.2, ls="--"), "one household (2+ homes)"),
        (Line2D([], [], color="#2B2B2B", lw=7), "main road"),
        (Line2D([], [], color="#FFD400", lw=3.5), "distance to the main road")]
    key.legend([h for h, _ in items], [l for _, l in items], loc="upper left", bbox_to_anchor=(0, 0.9), frameon=False, fontsize=13.5)
    finish(fig, "fig_v3_10_rules_closeup.png", "Source: Overture Maps, OpenStreetMap; imagery Esri World Imagery; model results")
    wc = gpd.GeoSeries([Point(*best)], crs=buildings.crs).to_crs(4326).iloc[0]
    print(f"CLOSE-UP centre lon {wc.x:.5f}, lat {wc.y:.5f}; arrow home is {target['dist_main_road_m']:.0f} m from the main road")
except Exception as err:
    print("fig_v3_10 failed:", err)

# ---------------------------------------------------------------- PART 12: PRINTOUTS (copy all of this back to Claude)
print("\n=== (a) Buildings and roof size (m²) by type ===")
print(pd.DataFrame({"t": btype, "a": buildings["area_m2"].to_numpy()}).groupby("t")["a"]
      .describe(percentiles=[0.25, 0.5, 0.75])[["count", "25%", "50%", "75%"]].round(0).to_string())
print("\n=== (b) Agriculture appliance list and greenhouse kWh ===")
print(APPLIANCES.get("agriculture", "no 'agriculture' list in APPLIANCES"))
ag = buildings.loc[buildings["building_type"] == "agriculture", "annual_kwh"]
if len(ag):
    print(f"{len(ag)} buildings: mean {ag.mean():,.0f}, min {ag.min():,.0f}, max {ag.max():,.0f} kWh/yr")
print("\n=== (c) Household kWh/yr by distance to a main road ===")
hhs = buildings[buildings["building_type"] == "residential"].groupby("household_id").agg(kwh=("annual_kwh", "sum"), d=("dist_main_road_m", "min"))
bands = pd.cut(hhs["d"], [0, 250, 500, 1000, np.inf], right=False, labels=["<250 m", "250-500 m", "500-1000 m", ">1 km"])
print(hhs.groupby(bands, observed=False)["kwh"].agg(["count", "mean"]).round(0).to_string())
print("\n=== (d) Productive use ===")
print(buildings.groupby(["productive_use", "productive_use_reason"]).size().to_string())
print("\n=== (e) Cooling category by type ===")
print(pd.crosstab(buildings["building_type"], buildings["cooling_category"]).to_string())
print(f"\n=== (f) Settings === MONTHLY_KWH_MEAN={MONTHLY_KWH_MEAN} | TEMP_SENSITIVITY={TEMP_SENSITIVITY} | ROAD_EFFECT={ROAD_EFFECT}")
print("\nAll figures are in:", FIG_DIR)

## Key numbers

A compact JSON summary of the whole run, saved to `building_table_out/key_numbers.json`.

In [ ]:
# Key numbers: a compact JSON summary of the whole run, saved to building_table_out/key_numbers.json.
import json

def _intdict(s):
    return {str(k): int(v) for k, v in s.items()}

def _floatdict(s, nd=1):
    return {str(k): round(float(v), nd) for k, v in s.items()}

area_peak_k = float(curve["total"].max())
area_peak_h = int(curve["total"].idxmax())
sum_peaks = float(buildings["peak_kw"].sum())
total_mwh = float(buildings["annual_kwh"].sum() / 1000)
homes_mwh = float(buildings.loc[buildings["profile_source"] == "household_readings", "annual_kwh"].sum() / 1000)

weather_sites = {}
for site, sub in weather_scenario.groupby("site"):
    cdd = float((sub["t2m"] - 18).clip(lower=0).sum())
    weather_sites[site] = {"cooling_degree_days": round(cdd, 0),
                           "daily_peak_kw_mean": round(float(sub["daily_peak_kw"].mean()), 0),
                           "daily_peak_kw_max": round(float(sub["daily_peak_kw"].max()), 0)}

key = {
    "total_buildings": int(len(buildings)),
    "by_outline_source": _intdict(buildings["outline_source"].value_counts()),
    "by_building_type": _intdict(buildings["building_type"].value_counts()),
    "labelled_buildings": _intdict(
        buildings.loc[buildings["type_source"].isin(["osm", "overture", "manual"]), "type_source"].value_counts()),
    "households": int(len(households)),
    "total_mwh_year": round(total_mwh, 1),
    "homes_share_of_mwh": round(homes_mwh / total_mwh, 3),
    "area_peak_kw": round(area_peak_k, 0),
    "area_peak_hour": area_peak_h,
    "sum_of_peaks_kw": round(sum_peaks, 0),
    "diversity_factor": round(sum_peaks / area_peak_k, 2),
    "avg_kwh_year_per_building_by_type": _floatdict(buildings.groupby("building_type")["annual_kwh"].mean(), 1),
    "main_roads": int(roads["is_main"].sum()),
    "median_dist_main_road_m": round(float(buildings["dist_main_road_m"].median()), 1),
    "household_kwh_by_road_band": _floatdict(by_road["avg_kwh_year"], 0),
    "weather_sites": weather_sites,
    "example_area": {"name": example_area["name"],
                     "counts": _intdict(example_area["counts"]),
                     "total_mwh_year": round(float(example_area["total_mwh"]), 1),
                     "peak_kw": round(float(example_area["peak"]), 1),
                     "peak_hour": int(example_area["peak_hour"])},
}

with open(os.path.join(WORK_DIR, "key_numbers.json"), "w") as f:
    json.dump(key, f, indent=2)
print(json.dumps(key, indent=2))
print("Saved:", os.path.join(WORK_DIR, "key_numbers.json"))

In [ ]:
# =====================================================================================
# FIGURES FOR THE MIDPOINT SLIDES — version 2 (each picture sized to its slide spot)
# Paste ALL of this into ONE new cell at the very BOTTOM of building_table_v5.ipynb
# (replace the cell you added last time), after "Run All" has finished. Then run it.
# It only reads the results and draws pictures; it changes nothing else.
# =====================================================================================
import os, textwrap, warnings
warnings.filterwarnings("ignore", message="The GeoSeries you are attempting to plot is empty")
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt, requests
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.transforms import blended_transform_factory
from shapely.geometry import box as _box, Point
from shapely.ops import nearest_points
try:
    import contextily as cx
except ImportError:
    cx = None
    print("contextily is not installed: maps will have no satellite photo. Fix: run  %pip install contextily  in a new cell, then re-run this cell.")

# ---------------------------------------------------------------- PART 1: SETUP
FIG_DIR = os.path.join(WORK_DIR, "figures_v3"); os.makedirs(FIG_DIR, exist_ok=True)
DPI = 200                                   # sharp on a projector; sizes below are in inches = slide size / 100
plt.rcParams.update({"font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "xtick.labelsize": 12,
                     "ytick.labelsize": 12, "legend.fontsize": 12, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
SITE_BBOX = (36.861042, -1.190780, 36.899746, -1.160067)        # the study box (W, S, E, N), degrees
NAMES = {"residential": "Homes", "outbuilding": "Outbuildings", "unknown_large": "Large, use unknown",
         "commercial": "Shops / businesses", "industrial": "Workshops", "education": "Schools",
         "religious": "Churches / mosques", "agriculture": "Greenhouses / farms", "health": "Clinics"}
COLORS = {"residential": "#4A76A8", "outbuilding": "#A9C4E2", "unknown_large": "#B8ADA6", "commercial": "#F28E2B",
          "industrial": "#76B7B2", "education": "#59A14F", "religious": "#B07AA1", "agriculture": "#EDC948",
          "health": "#E15759"}
ORDER = ["residential", "outbuilding", "unknown_large", "commercial", "industrial", "education", "religious",
         "agriculture", "health"]
GROUPS = [("Homes", ["residential"], "#4A76A8"),
          ("Shops, schools, clinics, churches", ["commercial", "industrial", "education", "health", "religious"], "#F28E2B"),
          ("Large unknown + greenhouses", ["unknown_large", "agriculture"], "#B8ADA6")]
HALO = [pe.withStroke(linewidth=3, foreground="white")]
SRC = "Source: Overture Maps 2026-09-23.1, OpenStreetMap, NASA POWER; model results"

def plain(t): return NAMES.get(t, str(t))
def col(t): return COLORS.get(t, "#999999")

def finish(fig, name, source=SRC):
    fig.text(0.006, 0.008, source, fontsize=7.5, color="#777777")
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)
    print("Saved:", path)

def add_satellite(ax, alpha=1.0, zoom="auto"):
    if cx is None:
        return
    for z in ([zoom, 17] if zoom != "auto" else ["auto"]):
        try:
            cx.add_basemap(ax, crs=buildings.crs, source=cx.providers.Esri.WorldImagery, attribution=False,
                           alpha=alpha, zoom=z, zorder=0)
            return
        except Exception as err:
            last = err
    print("  (no satellite photo:", str(last)[:80], ")")

def shade_day(ax, labels=True):
    ax.axvspan(8, 17, color="#EEEEEE", zorder=0); ax.axvspan(18, 22, color="#FFF2C6", zorder=0)
    if labels:
        tr = ax.get_xaxis_transform()
        ax.text(12.5, 1.01, "daytime", transform=tr, ha="center", va="bottom", color="#777777", fontsize=11)
        ax.text(20, 1.01, "evening", transform=tr, ha="center", va="bottom", color="#A07800", fontsize=11)

def scale_bar(ax, metres, label):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x, y = x0 + 0.04 * (x1 - x0), y0 + 0.05 * (y1 - y0)
    ax.plot([x, x + metres], [y, y], color="white", lw=7, solid_capstyle="butt", zorder=9)
    ax.plot([x, x + metres], [y, y], color="black", lw=3.5, solid_capstyle="butt", zorder=10)
    ax.text(x + metres / 2, y + 0.02 * (y1 - y0), label, ha="center", va="bottom", fontsize=12,
            path_effects=HALO, zorder=10)

def group_day(mask):
    """Average day (kW per hour) of the buildings in `mask`, added up into the 3 GROUPS."""
    return {g: hourly_by_building[mask & np.isin(btype, kinds)].sum(axis=0) for g, kinds, _ in GROUPS}

def stack_day(ax, days, peak_label=True):
    shade_day(ax)
    names = [g for g, _, _ in GROUPS if days[g].sum() > 0]
    ax.stackplot(range(24), [days[g] for g in names], colors=[c for g, _, c in GROUPS if g in names],
                 labels=names, alpha=0.95, zorder=2)
    total = sum(days[g] for g in names)
    ax.set_xlim(0, 23); ax.set_ylim(0, total.max() * 1.18); ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel("Hour of the day"); ax.set_ylabel("kW")
    ax.legend(loc="upper left", frameon=True, framealpha=0.95, fontsize=11)
    return total

assert len(hourly_by_building) == len(buildings), "Run the whole notebook again first (Run All)."
btype = buildings["building_type"].astype(str).to_numpy()
types_here = [t for t in ORDER if t in set(btype)] + sorted(set(btype) - set(ORDER))
centres = buildings.geometry.centroid
site_total = hourly_by_building.sum(axis=0)
AREA_PEAK, AREA_PEAK_HOUR = site_total.max(), int(site_total.argmax())
site_poly = gpd.GeoSeries([_box(*SITE_BBOX)], crs=4326).to_crs(buildings.crs).iloc[0]
SITE_KM2 = site_poly.area / 1e6
sx0, sy0, sx1, sy1 = site_poly.bounds
N_HH = int(buildings["household_id"].nunique())
NAME_COL = next((c for c in ["osm_name", "name", "place_name", "poi_name"] if c in buildings.columns), None)
counts = pd.Series(btype).value_counts()

# The example 500 m block (slide 8). Leave None to pick it automatically near "Jacaranda".
EXAMPLE_CENTER_LONLAT = None        # or e.g. (36.8930, -1.1850)
EX = 500
def _score(x, y):
    inside = centres.within(_box(x - EX / 2, y - EX / 2, x + EX / 2, y + EX / 2)).to_numpy()
    kinds = set(btype[inside]) & {"commercial", "education", "health", "religious", "industrial", "agriculture"}
    return len(kinds) * 1000 + np.isin(btype[inside], list(kinds)).sum() * 10 + inside.sum() / 100
if EXAMPLE_CENTER_LONLAT:
    p = gpd.GeoSeries([Point(*EXAMPLE_CENTER_LONLAT)], crs=4326).to_crs(buildings.crs).iloc[0]; EX_X, EX_Y = p.x, p.y
else:
    hit = buildings[buildings[NAME_COL].astype(str).str.contains("Jacaranda", case=False, na=False)] if NAME_COL else []
    if len(hit):
        a = hit.geometry.centroid.iloc[0]
        cands = [(a.x + dx, a.y + dy) for dx in range(-200, 201, 50) for dy in range(-200, 201, 50)]
    else:
        cands = [(x, y) for x in np.arange(sx0 + 250, sx1 - 250, 125) for y in np.arange(sy0 + 250, sy1 - 250, 125)]
    EX_X, EX_Y = max(cands, key=lambda c: _score(*c))
EX_BOX = _box(EX_X - EX / 2, EX_Y - EX / 2, EX_X + EX / 2, EX_Y + EX / 2)
in_block = centres.within(EX_BOX).to_numpy()
print(f"Site {SITE_KM2:.1f} km² | {len(buildings):,} buildings | {N_HH:,} households | peak {AREA_PEAK:,.0f} kW at {AREA_PEAK_HOUR}:00 "
      f"| {buildings['annual_kwh'].sum() / 1000:,.0f} MWh/yr")

# ---------------------------------------------------------------- PART 2: fig_v3_01_site_map (slides 1-2)
try:
    fig = plt.figure(figsize=(16, 9))
    ax = fig.add_axes([0.005, 0.03, 0.70, 0.95])
    for t in types_here:
        buildings[btype == t].plot(ax=ax, color=col(t), edgecolor="none", zorder=3)
    ax.set_xlim(sx0, sx1); ax.set_ylim(sy0, sy1); ax.set_anchor("W")
    add_satellite(ax)
    ax.plot(*EX_BOX.exterior.xy, color="white", lw=3, ls="--", zorder=5)
    ax.text(EX_BOX.bounds[0] + 15, EX_BOX.bounds[3] - 15, "Example\nblock", color="white", fontsize=13, va="top",
            fontweight="bold", path_effects=[pe.withStroke(linewidth=3, foreground="black")], zorder=6)
    scale_bar(ax, 500, "500 m"); ax.set_axis_off()
    key = fig.add_axes([0.72, 0.08, 0.27, 0.86]); key.set_axis_off()
    key.text(0, 1, "Kahawa West,\nNairobi–Kiambu border", fontsize=20, fontweight="bold", va="top")
    key.text(0, 0.86, f"{SITE_KM2:.1f} km² · {len(buildings):,} buildings", fontsize=15, va="top", color="#444444")
    for i, t in enumerate(types_here):
        y = 0.74 - i * 0.065
        key.add_patch(plt.Rectangle((0, y - 0.022), 0.07, 0.04, color=col(t), transform=key.transAxes))
        key.text(0.1, y, f"{plain(t)}  ({counts[t]:,})", fontsize=14, va="center")
    finish(fig, "fig_v3_01_site_map.png", "Source: Overture Maps 2026-09-23.1, OpenStreetMap; imagery Esri World Imagery; model results")
except Exception as err:
    print("fig_v3_01 failed:", err)

# ---------------------------------------------------------------- PART 3: fig_v3_02_counts (slide 6, left; 850 x 480)
try:
    st = (pd.DataFrame({"t": btype, "a": buildings["area_m2"].to_numpy()}).groupby("t")["a"]
          .agg(n="size", med="median").sort_values("n"))
    fig = plt.figure(figsize=(8.5, 4.8))
    ax = fig.add_axes([0.25, 0.1, 0.5, 0.76])
    y = np.arange(len(st))
    ax.barh(y, st["n"], color=[col(t) for t in st.index], height=0.7)
    for yi, n_ in zip(y, st["n"]):
        ax.text(n_ + st["n"].max() * 0.02, yi, f"{int(n_):,}", va="center", fontsize=12)
    ax.set_yticks(y); ax.set_yticklabels([plain(t) for t in st.index], fontsize=12.5)
    ax.set_xlim(0, st["n"].max() * 1.18); ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    tr = blended_transform_factory(ax.transAxes, ax.transData)
    ax.text(1.08, len(st) - 0.3, "typical roof", transform=tr, fontsize=11.5, color="#555555", fontweight="bold")
    for yi, m in zip(y, st["med"]):
        ax.text(1.08, yi, f"{m:,.0f} m²", transform=tr, va="center", fontsize=12, color="#333333")
    ax.set_title(f"Buildings by type  ({N_HH:,} households)", fontsize=14, loc="left", x=-0.45)
    finish(fig, "fig_v3_02_counts.png", "Source: Overture Maps, OpenStreetMap; model results. Typical roof = median.")
except Exception as err:
    print("fig_v3_02 failed:", err)

# ---------------------------------------------------------------- PART 4: fig_v3_03_area_day (slide 6, right; 850 x 480)
try:
    fig = plt.figure(figsize=(8.5, 4.8))
    ax = fig.add_axes([0.1, 0.17, 0.87, 0.7])
    tot = stack_day(ax, group_day(np.ones(len(buildings), bool)))
    mh = int(np.argmax(tot[4:11]) + 4)
    ax.annotate(f"Evening peak {tot.max():,.0f} kW", (AREA_PEAK_HOUR, tot.max()), (10.3, tot.max() * 1.0),
                fontsize=12, arrowprops=dict(arrowstyle="->"), va="center")
    ax.annotate(f"Morning {tot[mh]:,.0f} kW", (mh, tot[mh]), (9.3, tot[mh] * 0.72), fontsize=12,
                arrowprops=dict(arrowstyle="->"), va="center")
    ax.set_ylabel("kW, whole site")
    ax.set_title("Whole site, average day", fontsize=14, loc="left", pad=16)
    finish(fig, "fig_v3_03_area_day.png")
except Exception as err:
    print("fig_v3_03 failed:", err)

# ---------------------------------------------------------------- PART 5: fig_v3_04_grid (dashboard view; 900 x 720)
try:
    if "cells" in globals() and "annual_mwh" in cells.columns:
        grid = cells.to_crs(buildings.crs)
    else:
        g = pd.DataFrame({"x": centres.x // CELL_M, "y": centres.y // CELL_M, "k": buildings["annual_kwh"].to_numpy()})
        a = g.groupby(["x", "y"])["k"].sum().reset_index()
        grid = gpd.GeoDataFrame({"annual_mwh": a["k"] / 1000}, crs=buildings.crs, geometry=[
            _box(x * CELL_M, y * CELL_M, (x + 1) * CELL_M, (y + 1) * CELL_M) for x, y in zip(a["x"], a["y"])])
    fig = plt.figure(figsize=(9, 7.2))
    ax = fig.add_axes([0.01, 0.22, 0.98, 0.76])
    vmax = grid["annual_mwh"].quantile(0.98)
    grid.plot(ax=ax, column="annual_mwh", cmap="YlOrRd", vmin=0, vmax=vmax, alpha=0.82, edgecolor="white",
              linewidth=0.5, zorder=2)
    ax.set_xlim(sx0, sx1); ax.set_ylim(sy0, sy1)
    add_satellite(ax, alpha=0.6)
    ax.plot(*EX_BOX.exterior.xy, color="black", lw=3.5, zorder=5)
    ax.text(EX_X, EX_Y, "Example\nblock", ha="center", va="center", fontsize=12, fontweight="bold",
            path_effects=HALO, zorder=6)
    scale_bar(ax, 500, "500 m"); ax.set_axis_off()
    cax = fig.add_axes([0.2, 0.13, 0.6, 0.026])
    cb = fig.colorbar(plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(0, vmax)), cax=cax, orientation="horizontal")
    cb.set_label(f"Electricity used per year in each {CELL_M} m × {CELL_M} m square (MWh)", fontsize=12)
    finish(fig, "fig_v3_04_grid.png")
except Exception as err:
    print("fig_v3_04 failed:", err)

# ---------------------------------------------------------------- PART 6: fig_v3_05a_block_map (slide 8, left; 1100 x 760)
try:
    blk, bt = buildings[in_block], btype[in_block]
    nb = pd.Series(bt).value_counts()
    fig = plt.figure(figsize=(11, 7.6))
    ax = fig.add_axes([0.005, 0.03, 0.66, 0.95])
    for t in types_here:
        blk[bt == t].plot(ax=ax, color=col(t), edgecolor="white", linewidth=0.3, zorder=3)
    ax.set_xlim(*EX_BOX.bounds[0::2]); ax.set_ylim(*EX_BOX.bounds[1::2]); ax.set_anchor("W")
    add_satellite(ax, zoom=18)
    scale_bar(ax, 100, "100 m"); ax.set_axis_off()
    places = []
    if NAME_COL:
        nm = blk[blk[NAME_COL].notna() & (blk[NAME_COL].astype(str).str.strip() != "")
                 & ~blk["building_type"].isin(["residential", "outbuilding"])].copy()
        prio = {"health": 0, "education": 1, "religious": 2, "industrial": 3, "commercial": 4, "agriculture": 5, "unknown_large": 6}
        nm["p"] = nm["building_type"].map(prio).fillna(9)
        nm = nm.sort_values(["p", "area_m2"], ascending=[True, False]).drop_duplicates(NAME_COL).head(6)
        for i, (_, r) in enumerate(nm.iterrows(), 1):
            c = r.geometry.centroid
            ax.scatter(c.x, c.y, s=330, color="white", edgecolor="black", linewidth=1.5, zorder=7)
            ax.text(c.x, c.y, str(i), ha="center", va="center", fontsize=11, fontweight="bold", zorder=8)
            places.append((i, str(r[NAME_COL]), plain(r["building_type"])))
    key = fig.add_axes([0.68, 0.03, 0.31, 0.93]); key.set_axis_off()
    key.text(0, 1, "Building types", fontsize=14, fontweight="bold", va="top")
    y = 0.93
    for t in types_here:
        if nb.get(t, 0):
            key.add_patch(plt.Rectangle((0, y - 0.016), 0.06, 0.03, color=col(t), transform=key.transAxes))
            key.text(0.09, y, f"{plain(t)} ({nb[t]})", fontsize=12, va="center"); y -= 0.045
    if places:
        y -= 0.03; key.text(0, y, "Numbered places", fontsize=14, fontweight="bold", va="top"); y -= 0.06
        for i, name, kind in places:
            lines = textwrap.wrap(f"{name} ({kind.lower()})", 34)[:2]
            if y - 0.033 * len(lines) < 0.02:
                break
            key.text(0, y, f"{i}", fontsize=12, fontweight="bold", va="top")
            key.text(0.08, y, "\n".join(lines), fontsize=11, va="top"); y -= 0.033 * len(lines) + 0.018
    finish(fig, "fig_v3_05a_block_map.png", "Source: Overture Maps, OpenStreetMap, Overture Places; imagery Esri World Imagery")
except Exception as err:
    print("fig_v3_05a failed:", err)

# ---------------------------------------------------------------- PART 7: fig_v3_05b_block_day (slide 8, right; 620 x 360)
try:
    fig = plt.figure(figsize=(6.2, 3.6))
    ax = fig.add_axes([0.13, 0.17, 0.84, 0.68])
    tot = stack_day(ax, group_day(in_block))
    pk, ph = tot.max(), int(tot.argmax())
    ax.annotate(f"{pk:.0f} kW at {ph}:00", (ph, pk), (9, pk * 1.02), fontsize=11.5, arrowprops=dict(arrowstyle="->"), va="center")
    ax.legend().remove()
    ax.set_title("This block's average day", fontsize=13, loc="left", pad=14)
    finish(fig, "fig_v3_05b_block_day.png", "Model results")
    hh = buildings.loc[in_block & (btype == "residential"), "household_id"].nunique()
    print(f"BLOCK: { {k: int(v) for k, v in pd.Series(btype[in_block]).value_counts().items()} } | households {hh} | "
          f"{buildings.loc[in_block, 'annual_kwh'].sum() / 1000:.0f} MWh/yr | peak {pk:.0f} kW at {ph}:00 | "
          f"≈{int(np.ceil(pk / 0.8 / 5) * 5)} kVA")
except Exception as err:
    print("fig_v3_05b failed:", err)

# ---------------------------------------------------------------- PART 8: fig_v3_06_share_then_avg (slide 7, left; 1040 x 585)
try:
    nbld = buildings[buildings["building_type"] != "outbuilding"]
    g = nbld.groupby("building_type")["annual_kwh"].agg(total="sum", mean="mean", n="size")
    g["share"] = 100 * g["total"] / g["total"].sum()
    g = g.sort_values("share")
    fig = plt.figure(figsize=(10.4, 5.85))
    a1 = fig.add_axes([0.2, 0.12, 0.34, 0.7]); a2 = fig.add_axes([0.6, 0.12, 0.34, 0.7])
    y = np.arange(len(g))
    a1.barh(y, g["share"], color=[col(t) for t in g.index], height=0.7)
    for yi, v in zip(y, g["share"]):
        a1.text(v + 2, yi, f"{v:.1f}%", va="center", fontsize=12)
    a1.set_yticks(y); a1.set_yticklabels([plain(t) for t in g.index], fontsize=12.5)
    a1.set_xlim(0, 115); a1.set_xlabel("% of the site's yearly kWh")
    a1.set_title(f"1. Share of total use\n({g['total'].sum() / 1000:,.0f} MWh a year)", fontsize=13.5, loc="left")
    bars = a2.barh(y, g["mean"], color=[col(t) for t in g.index], height=0.7)
    for yi, (t, v) in zip(y, g["mean"].items()):
        a2.text(v + g["mean"].max() * 0.02, yi, f"{v:,.0f}" + ("*" if t == "agriculture" else ""), va="center", fontsize=12)
        if t == "agriculture":
            bars[list(g.index).index(t)].set_hatch("//"); bars[list(g.index).index(t)].set_edgecolor("#7A6A20")
    a2.set_yticks(y); a2.set_yticklabels([])
    a2.set_xlim(0, g["mean"].max() * 1.25); a2.set_xlabel("kWh per building per year")
    a2.set_title("2. Average individual building\nannual consumption", fontsize=13.5, loc="left")
    fig.suptitle(f"Homes use {g['share'].get('residential', 0):.0f}% of the electricity: each uses little, "
                 f"but there are {int(g['n'].get('residential', 0)):,} of them", fontsize=14.5, x=0.02, ha="left", y=0.97)
    finish(fig, "fig_v3_06_share_then_avg.png", "* placeholder appliance list. Homes: survey profiles (Nigeria); others: RAMP "
           "appliance lists (placeholders). Outbuildings are counted with their home.")
except Exception as err:
    print("fig_v3_06 failed:", err)

# ---------------------------------------------------------------- PART 9: fig_v3_07_type_shapes (slide 7, right; 656 x 585)
try:
    fig = plt.figure(figsize=(6.56, 5.85))
    ax = fig.add_axes([0.13, 0.25, 0.83, 0.6])
    shade_day(ax)
    shown = []
    for t in ["residential", "commercial", "education", "health"]:
        d = hourly_by_building[btype == t].sum(axis=0)
        if d.sum() > 0:
            ax.plot(range(24), 100 * d / d.sum(), color=col(t), lw=3, label=plain(t), zorder=3); shown.append(t)
    ax.set_xlim(0, 23); ax.set_xticks(range(0, 24, 3)); ax.set_xlabel("Hour of the day")
    ax.set_ylabel("% of that type's daily use")
    ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.17), ncol=2, frameon=False, fontsize=12)
    ax.set_title("Homes peak in the evening;\nshops, schools, clinics in the daytime", fontsize=13.5, loc="left", pad=18)
    finish(fig, "fig_v3_07_type_shapes.png", "Homes: survey profiles (Nigeria); others: RAMP (placeholders)")
except Exception as err:
    print("fig_v3_07 failed:", err)

# ---------------------------------------------------------------- PART 10: fig_v3_08_weather (slide 9; 1060 x 600)
try:
    def nasa_daily(lat, lon):
        prm = {"parameters": "T2M,T2M_MAX", "community": "RE", "longitude": lon, "latitude": lat,
               "start": f"{WEATHER_YEAR}0101", "end": f"{WEATHER_YEAR}1231", "format": "JSON"}
        r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", params=prm, timeout=120)
        r.raise_for_status(); p = r.json()["properties"]["parameter"]
        mean = pd.Series(p["T2M"], dtype=float).replace(-999, np.nan); mx = pd.Series(p["T2M_MAX"], dtype=float).replace(-999, np.nan)
        mean.index = mx.index = pd.to_datetime(mean.index, format="%Y%m%d")
        return mean, mx
    c4326 = gpd.GeoSeries([site_poly.centroid], crs=buildings.crs).to_crs(4326).iloc[0]
    s_mean, s_max = nasa_daily(round(c4326.y, 4), round(c4326.x, 4))
    m_mean, m_max = nasa_daily(-4.0435, 39.6682)                                  # Mombasa
    s_cdd, m_cdd = (s_mean - 18).clip(lower=0).sum(), (m_mean - 18).clip(lower=0).sum()
    base = s_mean.mean()
    s_pk = AREA_PEAK * (1 + TEMP_SENSITIVITY * (s_mean - base)); m_pk = AREA_PEAK * (1 + TEMP_SENSITIVITY * (m_mean - base))
    fig = plt.figure(figsize=(10.6, 6.0))
    a1 = fig.add_axes([0.07, 0.3, 0.4, 0.58]); a2 = fig.add_axes([0.58, 0.3, 0.4, 0.58])
    for ser, c in [(m_mean, "#D62728"), (s_mean, "#1F5FA8")]:
        a1.plot(ser.index, ser, color=c, lw=1.3); a1.fill_between(ser.index, 18, ser, where=ser > 18, color=c, alpha=0.25)
    a1.axhline(18, color="black", lw=1, ls="--")
    a1.text(s_mean.index[-1], 18.2, "18 °C", ha="right", va="bottom", fontsize=11, path_effects=HALO)
    a1.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 3, 5, 7, 9, 11])); a1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    a1.set_ylabel("Daily average temperature (°C)")
    a1.set_title(f"MEASURED ({WEATHER_YEAR}): days above 18 °C\nshaded area = cooling score", fontsize=13, loc="left")
    a1.legend(handles=[Line2D([], [], color="#1F5FA8", lw=3, label=f"This site: {s_cdd:,.0f} degree-days (≈{s_cdd / 365:.1f} °C a day)"),
                       Line2D([], [], color="#D62728", lw=3, label=f"Mombasa: {m_cdd:,.0f} degree-days (≈{m_cdd / 365:.1f} °C a day)")],
              loc="upper left", bbox_to_anchor=(-0.05, -0.1), frameon=False, fontsize=11.5)
    a2.scatter(s_max, s_pk, s=14, color="#1F5FA8", alpha=0.7); a2.scatter(m_max, m_pk, s=14, color="#D62728", alpha=0.7)
    a2.set_xlabel("Daily maximum temperature (°C)"); a2.set_ylabel("Modelled daily peak, whole site (kW)")
    a2.set_title("WHAT-IF: same buildings, hotter weather", fontsize=13, loc="left")
    a2.legend(handles=[Line2D([], [], marker="o", ls="", color="#1F5FA8", label=f"This site's weather (avg {s_pk.mean():,.0f} kW)"),
                       Line2D([], [], marker="o", ls="", color="#D62728", label=f"Mombasa's weather (avg {m_pk.mean():,.0f} kW)")],
              loc="upper left", bbox_to_anchor=(-0.05, -0.17), frameon=False, fontsize=11.5)
    a2.text(0.98, 0.03, f"Slope = our assumed +{TEMP_SENSITIVITY:.0%} demand per °C (placeholder)", fontsize=10, color="#555555",
            transform=a2.transAxes, ha="right")
    finish(fig, "fig_v3_08_weather.png", f"Source: NASA POWER daily temperatures, {WEATHER_YEAR}; peak = model results")
    print(f"WEATHER: site {s_cdd:.0f} CDD (mean {s_mean.mean():.1f} °C) | Mombasa {m_cdd:.0f} CDD (mean {m_mean.mean():.1f} °C) | "
          f"avg daily peak {s_pk.mean():.0f} -> {m_pk.mean():.0f} kW (+{m_pk.mean() / s_pk.mean() - 1:.0%})")
except Exception as err:
    print("fig_v3_08 failed:", err)

# ---------------------------------------------------------------- PART 11: fig_v3_10_rules_closeup (slide 10; 1728 x 560)
try:
    has_roads = "roads" in globals() and roads is not None and "is_main" in roads.columns
    W, H = 200, 88                                                           # window in metres (fits the slide strip)
    out_mask = buildings["building_type"] == "outbuilding"
    n_out = buildings.loc[out_mask, "main_building_id"].value_counts()
    if has_roads:
        mr = roads[roads["is_main"]].to_crs(buildings.crs)
        road_union = mr.geometry.union_all() if hasattr(mr.geometry, "union_all") else mr.geometry.unary_union
    cand = buildings[(buildings["building_type"] == "residential") & (buildings["productive_use"] == "Medium")
                     & buildings["dist_main_road_m"].between(30, 70)]
    if not len(cand):
        cand = buildings[(buildings["building_type"] == "residential") & buildings["dist_main_road_m"].between(30, 70)]
    best, best_s = None, -1
    for _, r in cand.head(300).iterrows():
        c = r.geometry.centroid
        rp = nearest_points(c, road_union)[1] if has_roads else c
        cxm, cym = (c.x + rp.x) / 2, (c.y + rp.y) / 2
        w = _box(cxm - W / 2, cym - H / 2, cxm + W / 2, cym + H / 2)
        ins = centres.within(w).to_numpy()
        s = 3 * buildings.loc[ins, "building_id"].isin(n_out[n_out >= 1].index).sum() + ins.sum() / 10
        if s > best_s:
            best, best_s, target, win = (cxm, cym), s, r, w
    fig = plt.figure(figsize=(17.28, 5.6))
    ax = fig.add_axes([0.003, 0.03, 0.66, 0.95])
    view = buildings[buildings.intersects(win)]
    vt = view["building_type"].astype(str)
    for t in types_here:
        view[vt == t].plot(ax=ax, color=col(t), edgecolor="white", linewidth=0.6, zorder=3)
    if has_roads:
        mr[mr.intersects(win.buffer(100))].plot(ax=ax, color="#2B2B2B", linewidth=9, zorder=2)
    geo = buildings.set_index("building_id").geometry
    for _, r in view[vt == "outbuilding"].iterrows():
        if pd.notna(r["main_building_id"]) and r["main_building_id"] in geo.index:
            a, b = r.geometry.centroid, geo[r["main_building_id"]].centroid
            ax.plot([a.x, b.x], [a.y, b.y], color="black", lw=2, zorder=4)
    for hid, grp in view[vt == "residential"].groupby("household_id"):
        if len(grp) >= 2:
            hull = grp.geometry.buffer(2.5).union_all() if hasattr(grp.geometry, "union_all") else grp.geometry.buffer(2.5).unary_union
            hull = hull.convex_hull
            ax.plot(*hull.exterior.xy, color="white", lw=2.2, ls="--", zorder=5)
    c = target.geometry.centroid
    if has_roads:
        rp = nearest_points(c, road_union)[1]
        ax.annotate("", xy=(rp.x, rp.y), xytext=(c.x, c.y), zorder=7,
                    arrowprops=dict(arrowstyle="-|>", lw=3.5, color="#FFD400", path_effects=[pe.withStroke(linewidth=6, foreground="black")]))
        ax.text((c.x + rp.x) / 2 + 4, (c.y + rp.y) / 2, f"{target['dist_main_road_m']:.0f} m to\nmain road", fontsize=13,
                fontweight="bold", path_effects=HALO, zorder=8, va="center")
    lbl = ("Large home near main road:\nproductive use Medium" if target["productive_use"] == "Medium"
           else f"Home, productive use {target['productive_use']}")
    ax.annotate(lbl, (c.x, c.y), (c.x - 0.33 * W, c.y - 0.3 * H), fontsize=12.5,
                path_effects=HALO, zorder=8, arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.set_xlim(*win.bounds[0::2]); ax.set_ylim(*win.bounds[1::2]); ax.set_anchor("W")
    add_satellite(ax, zoom=18); ax.set_axis_off()
    key = fig.add_axes([0.68, 0.06, 0.31, 0.9]); key.set_axis_off()
    key.text(0, 1, "How to read it", fontsize=15, fontweight="bold", va="top")
    items = [(Patch(color=col(t)), plain(t)) for t in types_here if (vt == t).any()] + [
        (Line2D([], [], color="black", lw=2), "outbuilding → the home it belongs to"),
        (Line2D([], [], color="#888888", lw=2.2, ls="--"), "one household (2+ homes)"),
        (Line2D([], [], color="#2B2B2B", lw=7), "main road"),
        (Line2D([], [], color="#FFD400", lw=3.5), "distance to the main road")]
    key.legend([h for h, _ in items], [l for _, l in items], loc="upper left", bbox_to_anchor=(0, 0.9), frameon=False, fontsize=13.5)
    finish(fig, "fig_v3_10_rules_closeup.png", "Source: Overture Maps, OpenStreetMap; imagery Esri World Imagery; model results")
    wc = gpd.GeoSeries([Point(*best)], crs=buildings.crs).to_crs(4326).iloc[0]
    print(f"CLOSE-UP centre lon {wc.x:.5f}, lat {wc.y:.5f}; arrow home is {target['dist_main_road_m']:.0f} m from the main road")
except Exception as err:
    print("fig_v3_10 failed:", err)

# ---------------------------------------------------------------- PART 12: PRINTOUTS (copy all of this back to Claude)
print("\n=== (a) Buildings and roof size (m²) by type ===")
print(pd.DataFrame({"t": btype, "a": buildings["area_m2"].to_numpy()}).groupby("t")["a"]
      .describe(percentiles=[0.25, 0.5, 0.75])[["count", "25%", "50%", "75%"]].round(0).to_string())
print("\n=== (b) Agriculture appliance list and greenhouse kWh ===")
print(APPLIANCES.get("agriculture", "no 'agriculture' list in APPLIANCES"))
ag = buildings.loc[buildings["building_type"] == "agriculture", "annual_kwh"]
if len(ag):
    print(f"{len(ag)} buildings: mean {ag.mean():,.0f}, min {ag.min():,.0f}, max {ag.max():,.0f} kWh/yr")
print("\n=== (c) Household kWh/yr by distance to a main road ===")
hhs = buildings[buildings["building_type"] == "residential"].groupby("household_id").agg(kwh=("annual_kwh", "sum"), d=("dist_main_road_m", "min"))
bands = pd.cut(hhs["d"], [0, 250, 500, 1000, np.inf], right=False, labels=["<250 m", "250-500 m", "500-1000 m", ">1 km"])
print(hhs.groupby(bands, observed=False)["kwh"].agg(["count", "mean"]).round(0).to_string())
print("\n=== (d) Productive use ===")
print(buildings.groupby(["productive_use", "productive_use_reason"]).size().to_string())
print("\n=== (e) Cooling category by type ===")
print(pd.crosstab(buildings["building_type"], buildings["cooling_category"]).to_string())
print(f"\n=== (f) Settings === MONTHLY_KWH_MEAN={MONTHLY_KWH_MEAN} | TEMP_SENSITIVITY={TEMP_SENSITIVITY} | ROAD_EFFECT={ROAD_EFFECT}")
print("\nAll figures are in:", FIG_DIR)

In [ ]:
# Redraw fig_v3_02_counts without the household count in the title
st = (pd.DataFrame({"t": btype, "a": buildings["area_m2"].to_numpy()}).groupby("t")["a"]
      .agg(n="size", med="median").sort_values("n"))
fig = plt.figure(figsize=(8.5, 4.8))
ax = fig.add_axes([0.25, 0.1, 0.5, 0.76])
y = np.arange(len(st))
ax.barh(y, st["n"], color=[col(t) for t in st.index], height=0.7)
for yi, n_ in zip(y, st["n"]):
    ax.text(n_ + st["n"].max() * 0.02, yi, f"{int(n_):,}", va="center", fontsize=12)
ax.set_yticks(y); ax.set_yticklabels([plain(t) for t in st.index], fontsize=12.5)
ax.set_xlim(0, st["n"].max() * 1.18); ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
tr = blended_transform_factory(ax.transAxes, ax.transData)
ax.text(1.08, len(st) - 0.3, "typical roof", transform=tr, fontsize=11.5, color="#555555", fontweight="bold")
for yi, m in zip(y, st["med"]):
    ax.text(1.08, yi, f"{m:,.0f} m²", transform=tr, va="center", fontsize=12, color="#333333")
ax.set_title(f"Buildings by type  ({len(buildings):,} in total)", fontsize=14, loc="left", x=-0.45)
finish(fig, "fig_v3_02_counts.png", "Source: Overture Maps, OpenStreetMap; model results. Typical roof = median.")